# [실습] 세그먼트 요율표 — 표본 적은 세그먼트 안정화

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 어제의 한 숫자에서 242개 세그먼트로

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">모형추정 심화통계 · Day 2</div>
<h1 style="color:#000000;margin:10px 0">세그먼트 요율표 — 표본 적은 세그먼트 안정화</h1>
<div style="font-size:1.05em;color:#656565">요율 산정 프로젝트(5일) 2일차 — 프랑스 자동차보험 계약 678,013건</div>
</div>

## 오늘 할 일

프랑스 자동차보험사의 **프라이싱(요율 산정) 분석가** 업무 둘째 날입니다. 어제는 우도가 최대가 되는 추정값(MLE) 1개를 계산했습니다. 오늘은 그 추정값에 회사의 과거 경험을 사전분포로 반영합니다.

오늘의 과업은 팀장이 요청한 한 문장에서 출발합니다 — **"우리 회사가 보유한 계약 전체의 연간 사고율을, 근거와 함께, 하나의 숫자로 보고하라."** 절 순서대로 목표 4개를 정리합니다.

1. **한 숫자** — 계약 678,013건의 연간 사고율을 노출로 보정해 한 값으로 계산합니다(미션 1). 이론 4절이 공식으로 산출한 0.1006건/년을 수치 최적화로 다시 계산해 소수점 아래 6자리까지 대조합니다.
2. **요율표** — 지역 × 브랜드 조합 세그먼트마다 사고율을 따로 적어 요율표 초안을 작성합니다(미션 2).
3. **수축** — 관측이 적은 세그먼트의 추정값을 감마 사전분포로 안정시킵니다(미션 3).
4. **유무 전환** — 측정 지표를 사고 건수에서 사고 유무로 바꿔 같은 절차를 한 번 더 적용하고, **지역 2개의 사후분포를 비교해** 어느 쪽의 사고 경험률이 더 높은지를 확률 1개로 답합니다(과제).

어제는 사고 발생 여부만 0/1 로 집계해, 마력까지 세분한 세그먼트에서 추정값의 불확실성이 큰 세그먼트를 구분했습니다. 오늘은 측정 지표를 **관측 기간으로 나눈 사고율**로 바꾸고, 세그먼트 기준도 **지역 × 브랜드**로 다시 정의해 그 조합부터 다시 집계합니다.

수업에서 함께 진행하는 것은 미션 1·2 와 정리이고, 미션 3 과 과제는 방과후에 수행합니다. 뒤로 미루더라도 정리를 먼저 읽으면 결론이 이어집니다. 코드 작성은 대부분 AI 에 위임하고, 우리는 **지시할 내용을 정하고 받은 결과를 확인하는 역할**을 담당합니다.

## 오늘의 데이터

**freMTPL2freq** 는 프랑스 자동차 책임보험(Motor Third-Party Liability)의 실제 계약 기록입니다. 계약 한 건이 한 행이고 678,013행 × 12컬럼이며, 공개 저장소 OpenML 에서 로그인 없이 내려받습니다. 어제 규칙을 고정해 둔 컬럼은 아래 2개이고, 뒤쪽 미션에서는 여기에 지역 `Region` 과 차량 브랜드 `VehBrand` 2개 열을 세그먼트 기준으로 추가해 사용합니다. 이론 4절에서 함께 다룬 마력 등급 `VehPower` 는 오늘 사용하지 않고, 이론 5절에서 이름만 언급한 도시화 등급 `Area` 는 형 변환과 첫 미리보기 표에만 등장하며 오늘 계산에는 사용하지 않습니다.

| 컬럼 | 뜻 | 단위 | 받자마자 고정하는 규칙 |
|---|---|---|---|
| `ClaimNb` | 보험 기간 중 사고(청구) 건수 | 건 | 상한 4건에서 절사 — 4건을 넘는 사고가 기록된 계약이 소수 포함되어 있어, 그 몇 건이 전체 평균을 왜곡하지 않게 하는 관례적 전처리입니다 |
| `Exposure` | 그 계약이 실제로 유지된 기간, 곧 **노출** | 년 | 상한 1.0년에서 절사 — 1년을 초과해 2년까지 기록된 계약이 소수 포함되어 있어 같은 이유로 절사합니다 |

계약 4건 중 3건이 1년을 채우지 못하고 종료되어 계약 1건의 평균 노출은 0.53년입니다. 사고 건수만 세면 관측 기간이 달라 비교가 성립하지 않습니다. 노출이 사고율의 분모가 되는 이유입니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Japanese_car_accident.jpg" width="640" height="360"/>
<p align="center"><em>▲ 교차로 접촉 사고 — 이 데이터의 각 행은 이런 사고 기록 1건에 대응한다. 계약별 사고 건수에서 공정한 보험료를 역산하는 것이 이번 실습의 목표</em><br><span style="font-size:0.9em;color:#656565">출처: Wikipedia, Japanese_car_accident.jpg</span></p>

## 이 노트북을 쓰는 법

**환경 안내** — ▶ 실행은 Google Colab 연결 안내를 따르고, AI 튜터는 상단 「Claude 도우미」를 설치한 뒤 사용합니다(온보딩 레슨 참고). **채점은 Colab 이 연결되면 자동으로 준비됩니다 — 코드 패널 상단의 「채점 준비됨」이 표시된 뒤 미션을 시작하고, 미션 셀을 실행했는데 채점 결과가 출력되지 않으면 그 「채점 준비됨」을 누르세요.**

노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다(코드는 접혀 있어도 실행됩니다). 도우미 대신 다른 AI 도구에 PROMPT 를 전달해 받은 코드를 `# --- 여기부터 AI 코드 ---` 아래에 직접 붙여 넣습니다. 채점이 끝나 🔓 가 표시되면 그 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 비교합니다.

**맨 위 설치·폰트 셀을 먼저 실행하세요 — 1회만 실행하면 되고, 채점 셀은 LMS 가 자동으로 실행합니다.** 미션마다 코드 셀이 2개입니다. 앞의 것은 그대로 실행하는 **준비 셀**이고, 뒤의 것은 지시할 내용을 적는 **PROMPT 셀**입니다.

PROMPT 셀의 `PROMPT` 에 지시할 내용을 직접 작성해 AI 에 전달하면, `# --- 여기부터 AI 코드 ---` 아래가 AI 가 코드를 작성하는 위치가 됩니다.

셀을 실행하면 **실행 결과(출력)에 채점 결과**가 함께 표시됩니다. 기준을 충족하면 완성본 풀이가 열리고, 미달이면 미달한 항목과 AI 튜터에 보낼 질문이 표시됩니다. 변수 이름은 자유롭게 정해도 됩니다. 채점 기준은 변수 이름이 아니라 셀이 출력한 값과 그림이기 때문입니다.

「준비 셀」은 미션마다 사용할 변수를 처음부터 정의하므로, 맨 위 설치·폰트 셀을 1회 실행해 두었다면 뒤쪽 미션만 따로 열어도 그 셀부터 실행하면 됩니다.

**주고받기 규칙 2개** — AI 와의 대화가 진전되지 않을 때는 다음 2개를 적용합니다.

| 이럴 때 | 이렇게 합니다 |
|---|---|
| 같은 요구를 두 번째 되풀이하게 될 때 | 표현을 바꾸지 말고 `PROMPT` 에 **누락된 조건을 1개** 추가합니다 |
| 조건을 추가해도 해결되지 않을 때 | 앞 대화를 이어서 사용하지 말고, 요구 사항 전체를 한 번에 다시 작성해 새 메시지로 전달합니다 |

In [ ]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "scipy>=1.11" "matplotlib>=3.7" "seaborn>=0.13" "scikit-learn>=1.3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats
from scipy.optimize import minimize
from scipy.special import gammaln
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 폰트 목록에는 그 설치가 반영되지 않으므로 addfont 로
#  등록해야 이름이 보인다(등록하지 않으면 DejaVu Sans 로 대체되어 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 축소되지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150      # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True                            # 여백은 tight_layout 이 자동 조정
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     # 위 테두리·오른쪽 테두리·격자를 제거해 데이터만 남긴다
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 4개가 함께 쓰는 색 — 사전=남색, 우도=파랑, 사후·강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3", "참값": "#757575"}
FIG_W = 10.0     # 캔버스 폭은 4개 모두 이 값으로 고정하고 높이만 그림마다 조절한다

print("사용 폰트:", FONT,
      "| 색 팔레트 준비 완료 — 채점 셀은 LMS 가 자동으로 준비합니다")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 받아 바로 아래에 채점 결과를 출력합니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 코드 행들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
           "eyJtMSI6IHsidmFsdWVzIjogeyLsl7DqsIQg7IKs6rOg7JyoIM67zIIgKOqxtC/rhYQpIjogWzAuMTAwMTEsIDAuMTAxMTIsICLq"
           "sbQv64WEIl0sICLstJ0g64W47LacICjrhYQpIjogWzM1ODAwMC4wLCAzNTg4MDAuMCwgIuuFhCJdfSwgInBsb3QiOiB7ImtpbmQi"
           "OiAibGluZSIsICJuX3NlcmllcyI6IDEsICJ4X3JhbmdlIjogWzAuMDUsIDAuMl0sICJwZWFrX3giOiBbMC4wOTUsIDAuMTA2XSwg"
           "InZsaW5lX3giOiBbMC4wOTUsIDAuMTA2XSwgIuy2leudvOuyqCI6IHRydWV9fSwgIm0yIjogeyJ2YWx1ZXMiOiB7IuyEuOq3uOuo"
           "vO2KuCDsiJggKOqwnCkiOiBbMjQxLjUsIDI0Mi41LCAi6rCcIl0sICLqsIDsnqUg64aS7J2AIOyEuOq3uOuovO2KuOydmCDsgqzq"
           "s6DsnKggKOqxtC/rhYQpIjogWzAuMzA3NjQsIDAuMzEwNzQsICLqsbQv64WEIl19LCAicGxvdCI6IHsia2luZCI6ICJzY2F0dGVy"
           "IiwgIm5fcG9pbnRzIjogWzI0MiwgMzEwXSwgInhzY2FsZSI6ICJsb2ciLCAieF9yYW5nZSI6IFsyLjQyLCAzNTk0OS4wXSwgInlt"
           "YXgiOiBbMC4zLCAwLjMyXSwgIuy2leudvOuyqCI6IHRydWV9fSwgIm0zIjogeyJ2YWx1ZXMiOiB7IlIyMSDDlyBCMTEg7IS46re4"
           "66i87Yq47J2YIE1BUCAo6rG0L+uFhCkiOiBbMC4xMTI1NiwgMC4xMTM2OSwgIuqxtC/rhYQiXSwgIjI0MuqwnCDshLjqt7jrqLzt"
           "irggTUFQIOydmCDstZzshp/qsJIgKOqxtC/rhYQpIjogWzAuMDU0NzksIDAuMDU1MzQsICLqsbQv64WEIl19LCAicGxvdCI6IHsi"
           "a2luZCI6ICJzY2F0dGVyIiwgIm5fcG9pbnRzIjogWzI0MiwgMzEwXSwgImRpYWciOiB0cnVlLCAieV9yYW5nZSI6IFswLjA1LCAw"
           "LjJdLCAiZ2FwX21pbiI6IDAuMTUsICLstpXrnbzrsqgiOiB0cnVlfX0sICLqs7zsoJwiOiB7InZhbHVlcyI6IHsiTUxFIOuhnCDq"
           "s6Drpbgg7IOB7JyEIOyXtCDshLjqt7jrqLztirjsnZgg6rOE7JW9IOyImCAo6rG0KSI6IFsyMzguNSwgMjM5LjUsICLqsbQiXSwg"
           "Ik1BUCDroZwg6rOg66W4IOyDgeychCDsl7Qg7IS46re466i87Yq47J2YIOqzhOyVvSDsiJggKOqxtCkiOiBbODg3MC4wLCA4ODkw"
           "LjAsICLqsbQiXSwgIuustOyCrOqzoCDtmZjquIkg64yA7IOBIOqzhOyVvSDsiJggKOqxtCkiOiBbNjQzODAwLjAsIDY0NDEwMC4w"
           "LCAi6rG0Il0sICJSNDIg6rCAIFI0MyDrs7Tri6Qg64aS7J2EIO2ZleuloCAo7ZmV66WgKSI6IFswLjk1NSwgMC45OTUsICLtmZXr"
           "paAiXX0sICJwbG90IjogeyJraW5kIjogInNjYXR0ZXIiLCAibl9wb2ludHMiOiBbMjQyLCAzMTBdLCAiZGlhZyI6IHRydWUsICJ4"
           "X3JhbmdlIjogWzAuMCwgMC4xNjY3XSwgInlfcmFuZ2UiOiBbMC4wMjQsIDAuMDg2OF0sICJnYXBfbWluIjogMC4wOCwgIuy2leud"
           "vOuyqCI6IHRydWV9fX0=")
_REVEAL = {"m1":                   # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
           "8J+UkyDrr7jshZggMSDsmYTshLHrs7gg4oCUIOOAjOyXrOq4sOu2gO2EsCBBSSDsvZTrk5zjgI0g7JWE656Y7JeQIOuTpOyWtOqwi"
           "CDsvZTrk5wg7JiI7Iuc7J6F64uI64ukCgrilqAg7J20IOy9lOuTnOulvCDrtoDrpbgg7ZSE66Gs7ZSE7Yq4IOyYiAoKICAgIOydtO"
           "uvuCDrp4zrk6TslrTsoLgg7J6I64qUIHksIEUsIHBvaXNzb25fbmxsIOydhCDqt7jrjIDroZwg7JOw6rOgIOuLpOyLnCDrp4zrk6T"
           "sp4DripQg66eQ7JWEIOykmC4g6rOE7JW9IOyghOyytOulvCDtlZjrgpjsnZgg7KeR64uo7Jy866GcIOuRkOqzoCBwb2lzc29uX25s"
           "bCDsnYQg6rCA7J6lIOyekeqyjCDrp4zrk5zripQg7Jew6rCECiAgICDsgqzqs6DsnKgg7ZWY64KY66W8IOyImOy5mCDstZzsoIHtm"
           "ZTroZwg7LC+7JWEIOykmC4g7LC+7J2AIOqwkuydgCAi7Jew6rCEIOyCrOqzoOycqCA9ICjsiKvsnpApIOqxtC/rhYQiIOyymOufvC"
           "DsnbTrpoTqs7wg64uo7JyE66W8IO2VqOq7mCDsoIHslrQg7IaM7IiY7KCQIOyVhOuemCA27J6Q66as6rmM7KeAIHByaW50IO2VmOq"
           "zoCwgRSDrpbwg7KCE67aAIOuNlO2VnCDstJ0KICAgIOuFuOy2nOuPhCAi7LSdIOuFuOy2nCA9ICjsiKvsnpApIOuFhCIg7Jy866Gc"
           "IO2VnCDtlokg642UIHByaW50IO2VtCDspJguIOy1nOygge2ZlOqwgCDsiJjroLTtlojripTsp4DsmYAg67CY67O1IO2an+yImCwg6"
           "re466as6rOgIOywvuydgCDsgqzqs6DsnKjsl5Ag7LSdIOuFuOy2nOydhCDqs7HtlZwg6rCS7J2EIOyLpOygnCDsgqzqs6Ag6rG07I"
           "iY7JmAIOuCmOuegO2eiAogICAg7ZGc7Iuc7ZWY64qUIOqygOyCsCDtlZwg7ZaJ64+EIOy2nOugpe2VtCDso7zqs6AuIOq3uOuemO2"
           "UhOuKlCDrlLEg7ZWcIOyepeunjCDqt7jroKQg7KSYIOKAlCDqsIDroZzstpXsl5Ag7IKs6rOg7JyoIO2bhOuztOulvCAwLjA1IOu2"
           "gO2EsCAwLjIwIOq5jOyngCDstJjstJjtnogg67Cw7LmY7ZWY6rOgIOyEuOuhnOy2leydgCDqt7gg7KeA7KCQ7J2YCiAgICDroZzqt"
           "7jsmrDrj4QocG9pc3Nvbl9ubGwg7JeQIOuniOydtOuEiOyKpOulvCDqs7HtlZwg6rCSKeuhnCDqs6HshKDsnYQg7ZWY64KY66eMIO"
           "q3uOumsCDrkqQsIOuwqeq4iCDssL7snYAg7IKs6rOg7JyoIOychOy5mOyXkCDsiJjsp4HshKDsnYQg7ZWY64KYIOq4i+qzoCDqsID"
           "roZzstpXqs7wg7IS466Gc7LaV7JeQIOudvOuyqOydhCDri6zslYQg7KSYLgoKICAgIHJlcyA9IG1pbmltaXplKGZ1bj1wb2lzc29u"
           "X25sbCwgeDA9WzAuNV0sIG1ldGhvZD0iTC1CRkdTLUIiLCBib3VuZHM9WygxZS02LCBOb25lKV0pCiAgICBsYW1faGF0ID0gZmxvY"
           "XQocmVzLnhbMF0pCiAgICBwcmludChmIuy0nSDrhbjstpwgICAgIDoge0Uuc3VtKCk6LC4xZn0g64WEIikKICAgIHByaW50KGYi7J"
           "ew6rCEIOyCrOqzoOycqCA6IHtsYW1faGF0Oi42Zn0g6rG0L+uFhCIpCiAgICBwcmludChmIuqygOyCsCA6IGxhbV9oYXQgeCDstJ0"
           "g64W47LacID0ge2xhbV9oYXQgKiBFLnN1bSgpOiwuMGZ9IOqxtCAo7Iuk7KCcIOyCrOqzoCB7eS5zdW0oKTosLjBmfSDqsbQpIikK"
           "ICAgIHByaW50KCLsiJjroLQiLCByZXMuc3VjY2VzcywgInwg67CY67O1IiwgcmVzLm5pdCwgIu2ajCB8IOuhnOq3uOyasOuPhCDst"
           "ZzrjJPqsJIg7KeA7KCQ7J2YIE5MTCIsIGYie3Jlcy5mdW46LC4xZn0iKQoKICAgIGdyaWQgPSBucC5saW5zcGFjZSgwLjA1LCAwLj"
           "IwLCAyMDApICAgICAgICAgICAgICAgICAgIyDsgqzqs6DsnKgg7ZuE67O0IDIwMOqwnAogICAgbG9nbGlrID0gbnAuYXJyYXkoWy1"
           "wb2lzc29uX25sbCh2KSBmb3IgdiBpbiBncmlkXSkgICAjIO2bhOuztOuniOuLpOydmCDroZzqt7jsmrDrj4QgKE5MTCDsnZgg67aA"
           "7Zi466W8IOuSpOynkeyKteuLiOuLpCkKICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oOCwgMy40KSkKICAgIGF4L"
           "nBsb3QoZ3JpZCwgbG9nbGlrKQogICAgYXguYXh2bGluZShsYW1faGF0LCBscz0iLS0iLCBjb2xvcj0iZ3JheSIpICAgICAgICAgIC"
           "AjIOuhnOq3uOyasOuPhOqwgCDstZzrjIDsnbgg7JyE7LmY7J2YIOyImOyngeyEoAogICAgYXguc2V0X3hsYWJlbCgi7Jew6rCEIOy"
           "CrOqzoOycqCDOuyAo6rG0L+uFhCkiKQogICAgYXguc2V0X3lsYWJlbCgi66Gc6re47Jqw64+EIOKEkyjOuykiKQogICAgcGx0LnNo"
           "b3coKQoK7ZW07ISdCgotIGBtaW5pbWl6ZWAg64qUIOyLnOyekeygkOydhCDsnbzrtoDrn6wg66i8IDAuNSDroZwg7ISk7KCV7ZaI6"
           "4qU642w64+EIDftmowg67CY67O1IOunjOyXkCDOu8yCIOKJiCAwLjEwMDYg7JeQIOyImOugtO2VqeuLiOuLpC4g66Gc6re47Jqw64"
           "+E6rCAIOy1nOuMk+qwkiDtlZjrgpjrv5Dsnbgg66ek64GI7ZWcIOqzoeyEoOydtOudvCDsi5zsnpHsoJDqs7wg66y06rSA7ZWY6rK"
           "MIOqwmeydgCDtlbTroZwg7IiY66C07ZWp64uI64ukLgotIOqwmeydgCDqsJLsnbQg6rO17Iud7Jy866GcIOuwlOuhnCDqtaztlZjr"
           "ipQg7ZW0KGNsb3NlZC1mb3JtKSBgeS5zdW0oKSAvIEUuc3VtKClgIOuhnOuPhCDqs4TsgrDrkKnri4jri6Qo65GQIOqwkuydmCDss"
           "KjsnbQg7JW9IDRlLTA5KS4g7IiY7LmYIOy1nOygge2ZlOqwgCDsoJzrjIDroZwg7IiY7ZaJ65CQ64qU7KeA64qUIOydtOyymOufvC"
           "Dri6Trpbgg67Cp67KV7Jy866GcIDHtmowg642UIOqzhOyCsO2VtCDruYTqtZDtlZjripQg6rKD7J20IOqwgOyepSDruaDrpbgg7Zm"
           "V7J247J6F64uI64ukLgotIOu2hOuqqOuKlCDqs4Tslb0g7IiYIDY3OCwwMTMg7J20IOyVhOuLiOudvCDstJ0g64W47LacIDM1OCwz"
           "NjDrhYTsnoXri4jri6QuIOqzhOyVvSDsiJjroZwg64KY64iE66m0IDAuMDUzMiDrnbzripQg6re465+065Ov7ZWcIOyIq+yekOqwg"
           "CDrgpjsmLXri4jri6QuIOuLqOychOu2gO2EsCDqsbQv6rOE7JW97J206528IOyalOycqOyXkCDsk7gg6rG0L+uFhOydtCDslYTri4"
           "jqs6Ag6rCS64+EIOygiOuwmCDsiJjspIDsnLzroZwg64Ku7JWE7KeR64uI64ukLiDsmKTrpZgg7JeG7J20IOqwkuunjCDti4Drpqz"
           "ripQg6rK97Jqw6528IOyxhOygkOydmCDssqsg7ZWt66qp7J20IOydtCDqsJLsnYQg6rKA7IKs7ZWp64uI64ukLgotIOuhnOq3uOya"
           "sOuPhOqwgCDstZzrjIDsnbgg7KeA7KCQ7J2YIE5MTCAxNDYsNjI0IOuKlCDsoovqs6Ag64KY7IGo7J2YIOygiOuMgCDquLDspIDsn"
           "bQg7JWE64uZ64uI64ukLiDqs4Tslb0gNjc4LDAxM+qxtOydmCDroZzqt7jtmZXrpaDsnYQg66qo65GQIOuNlO2VnCDqsJLsnbTrnb"
           "wg7ZGc67O47J20IOy7pOyngOuptCDtlajqu5gg7Luk7KeR64uI64ukLiDqsJnsnYAg642w7J207YSw7JeQ7IScIO2bhOuztOulvCD"
           "ruYTqtZDtlaAg65WM66eMIOydmOuvuOqwgCDsnojsirXri4jri6QoMC4wNSDsnZggMTUzLDY5OSDrjIAgMC4xMCDsnZggMTQ2LDYy"
           "NSkuCi0g7JqU7JyoIOuztOqzoOyEnCDssqsg66y47J6l7J2AIOydtOugh+qyjCDslIHri4jri6Qg4oCUICLri7nsgqwg7Y+s7Yq47"
           "Y+066as7Jik7J2YIOyXsOqwhCDsgqzqs6DsnKjsnYAg64W47LacIDHrhYTri7kgMC4xMDA26rG0KD0g64W47LacIDEwMOuFhOuLuS"
           "Dslb0gMTDqsbQp7Jy866GcIOy2lOygleuQqC4g6re86rGwOiDrhbjstpwg67O07KCVIO2PrOyVhOyGoSDrqqjtmJXsnZgg7LWc64y"
           "A7Jqw64+E7LaU7KCVLCDqs4Tslb0gNjc4LDAxM+qxtC4iCgrsvZTrk5zqsIAg64u17J2EIOunjOuTnOuKlCDrsKnsi50g4oCUIOy0"
           "nSDsgqzqs6Ag6rG07IiY66W8IOy0nSDrhbjstpzroZwg64KY64iV64uI64ukKD0g64W47LacIOuztOyglSDtj6zslYTshqHsnZgg7"
           "J2M7J2YIOuhnOq3uOyasOuPhOulvCDqsIDsnqUg7J6R6rKMIOunjOuTnOuKlCDOuyDrpbwg7LC+7Iq164uI64ukKS4K",
           "m2":
           "8J+UkyDrr7jshZggMiDsmYTshLHrs7gg4oCUIOqzoOy5oCDqs7PsnYAg44CM7Jes6riw67aA7YSwIEFJIOy9lOuTnOOAjSDslYTr"
           "npgg7L2U65OcIO2VnCDtlonsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg"
           "7KCE7J6EIOuLtOuLueyekOqwgCDrp4zrk6TslrQg65GUIDI0MuqwnCDshLjqt7jrqLztirgg7JqU7Jyo7ZGcIHNlZ19maW5lIOyd"
           "gCDqt7jrjIDroZwg65GQ6rOgLCDshLjqt7jrqLztirjrs4Qg7IKs6rOg7JyoIGxhbV9tbGUg7J2EIOunjOuTnOuKlCDrgpjriJfs"
           "hYjsnZgg67aE66qo66eMIOqzoOyzkCDspJguIOyngOq4iOydgCDqt7gg7IS46re466i87Yq47J2YIOqzhOyVvSDsiJggbiDsnLzr"
           "oZwKICAgIOuCmOuIhOqzoCDsnojripTrjbAg7IKs6rOg7Jyo7J2YIOu2hOuqqOuKlCDqt7gg7IS46re466i87Yq47J2YIOuFuOy2"
           "nCDtlansnbggZXhwb3N1cmUg7Je07J206528IOuLqOychOqwgCDqsbQv64WE7J20IOuQmOyWtOyVvCDtlbQg4oCUIHNlZ19maW5l"
           "IOyXkOuKlCDshLjqt7jrqLztirgg7J2066aEIFJlZ2lvbiDCtyBWZWhCcmFuZCDsmYAKICAgIOqzhOyVvSDsiJggbiDCtyDsgqzq"
           "s6Ag6rG07IiYIGNsYWltcyDCtyDrhbjstpwgZXhwb3N1cmUg7Je07J20IOyeiOycvOuLiCBjbGFpbXMg66W8IGV4cG9zdXJlIOuh"
           "nCDrgpjriIgg6rCS7J2EIGxhbV9tbGUg7Je07JeQIOuLpOyLnCDrhKPslrQg7KSYLiDsp5Hqs4TripQg7J2066+4CiAgICDrgZ3r"
           "gpgg7J6I7Jy864uIIGdyb3VwYnkg66W8IOuLpOyLnCDsi6TtlontlZjsp4DripQg66eQ6rOgLiDqs6DsuZwg65KkIOyalOycqO2R"
           "nOydmCDshLjqt7jrqLztirgg7IiY66W8ICLshLjqt7jrqLztirgg7IiYID0gKOyIq+yekCkg6rCcIiDroZwsIOqwgOyepSDrhpLs"
           "nYAg7IS46re466i87Yq47J2YIOyXsOqwhCDsgqzqs6DsnKjsnYQgIuqwgOyepSDrhpLsnYAg7IS46re466i87Yq47J2YIOyCrOqz"
           "oOycqCA9CiAgICAo7Iir7J6QKSDqsbQv64WEIiDsspjrn7wg7IaM7IiY7KCQIOyVhOuemCA27J6Q66as6rmM7KeAIOuLqOychOyZ"
           "gCDtlajqu5ggcHJpbnQg7ZWY6rOgIOq3uCDshLjqt7jrqLztirjsnZggUmVnaW9uIHggVmVoQnJhbmQg7J2066aE64+EIOqwmeyd"
           "gCDtlonsl5Ag7ZWo6ruYIOyggeyWtCDspJguIOq3uOumvOydgCDsoITsnoQg64u064u57J6QIOy9lOuTnOyXkCDsl4bsnLzri4gK"
           "ICAgIOyDiOuhnCDqt7jroKQg7KSYIOKAlCDsoJAg7ZWY64KY6rCAIOyEuOq3uOuovO2KuCDtlZjrgpjsnbgg7IKw7KCQ64+E66Gc"
           "LCDqsIDroZzstpXsnYAg6re4IOyEuOq3uOuovO2KuOydmCDstJ0g64W47LacIGV4cG9zdXJlKOuFhCnsnYQg66Gc6re4IOuIiOq4"
           "iOyXkCDrsLDsuZjtlZjqs6Ag7IS466Gc7LaV7J2AIOq3uCDshLjqt7jrqLztirjsnZgg7Jew6rCEIOyCrOqzoOycqAogICAgbGFt"
           "X21sZSjqsbQv64WEKeydhCDrsLDsuZjtlZwg65KkIOuRkCDstpXsl5Ag652867Ko7J2EIOuLrOyVhCDspJguCgogICAgc2VnX2Zp"
           "bmVbImxhbV9tbGUiXSA9IHNlZ19maW5lWyJjbGFpbXMiXSAvIHNlZ19maW5lWyJleHBvc3VyZSJdICAgIyDrtoTrqqjripQg6re4"
           "IOyEuOq3uOuovO2KuOydmCDrhbjstpwg7ZWpICjrhYQpCgogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg4LCAz"
           "LjQpKSAgICAgICAgICAgICAgICAgICAgICAgICAgIyDqt7jrprzsnYAg7IOI66GcIOyLnO2CqCDrtoDrtoQKICAgIGF4LnNjYXR0"
           "ZXIoc2VnX2ZpbmVbImV4cG9zdXJlIl0sIHNlZ19maW5lWyJsYW1fbWxlIl0sIHM9MTQpCiAgICBheC5zZXRfeHNjYWxlKCJsb2ci"
           "KSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIOuFuOy2nOydtCAyLjTrhYQgfiAz66eMIOuF"
           "hOydtOudvCDroZzqt7gg64iI6riICiAgICBheC5zZXRfeGxhYmVsKCLshLjqt7jrqLztirgg7LSdIOuFuOy2nCAo64WELCDroZzq"
           "t7gg64iI6riIKSIpCiAgICBheC5zZXRfeWxhYmVsKCLsl7DqsIQg7IKs6rOg7JyoIE1MRSAo6rG0L+uFhCkiKQogICAgcGx0LnNo"
           "b3coKQoK7ZW07ISdCgotIOynkeqzhOuKlCDsspjsnYzrtoDthLAg66ee7JWY7Iq164uI64ukIOKAlCDshLjqt7jrqLztirggMjQy"
           "6rCc64+ELCDshLjqt7jrqLztirjrs4Qg7IKs6rOgIO2VqSAzNiwwNTbqsbQoPSDsm5Drs7ggY2xhaW1zX2FsbCnrj4Qg66ee7JWY"
           "7Iq164uI64ukLiDti4DrprAg6rKD7J2AIOuCmOuIl+yFiOydmCDrtoTrqqgg7ZWcIOqzs+u/kOydtOudvCDsmKTrpZjqsIAg64KY"
           "7KeAIOyViuyVmOyKteuLiOuLpC4g7J207LKY65+8IOyYpOulmCDsl4bsnbQg6rCS66eMIO2LgOumrOuKlCDsvZTrk5zripQg7Iuk"
           "7ZaJIOqysOqzvOqwgCDslYTri4jrnbwg64uo7JyEIOqygOyCsOycvOuhnCDqsoDstpztlanri4jri6QuCi0g6rOE7JW9IOyImOuh"
           "nCDrgpjriIgg7ZGc64qUIOyEuOq3uOuovO2KuOuzhCDqsJLsnYQgMC4wMDAwMDAgfiAwLjE2NjY2NyDroZwg67CY7ZmY7ZWp64uI"
           "64ukLiDsnbQg7Iir7J6Q7J2YIOuLqOychOuKlCAi6rOE7JW9IO2VnCDqsbTsnbQg6rSA7Lih65CcIOq4sOqwhCDrj5nslYgg64K4"
           "IOyCrOqzoCDsiJgiKOqxtC/qs4Tslb0p6528IOyalOycqOyXkCDsk7gg6rG0L+uFhOydtCDslYTri5nri4jri6QuIOuFuOy2nOuh"
           "nCDqsIDspJHtlbQg64uk7IucIO2Vqey5mOuptCAwLjA1NDAg7J20IOuCmOyZgCDsoITssrQg6rOE7JW9IOyCrOqzoOycqCBsYW1f"
           "YWxsIOqzvCDsnbzsuZjtlZjsp4Ag7JWK64qUIOqyg+ydtCDqt7gg7Kad6rGw7J6F64uI64ukKOqzoOy5nCDtkZzripQgMC4xMDA2"
           "MTQg66GcIOygle2Zle2eiCDsnbzsuZjtlanri4jri6QpLgotIOu2hOuqqOulvCDqs6DsuZwg7ZGc7JeQ7IScIOyCrOqzoOycqOyd"
           "gCAwLjAwMDAwMCDrtoDthLAgMC4zMDkxOTEg6rG0L+uFhOq5jOyngCDrtoTtj6ztlZjqs6AsIDAg7Jy866GcIOqzhOyCsOuQnCDs"
           "hLjqt7jrqLztirjqsIAgOeqwnCDrgpjsmLXri4jri6QuIOqwgOyepSDrhpLsnYAg7IS46re466i87Yq4IFI0MyDDlyBCNCDripQg"
           "6rOE7JW9IDI26rG0wrfrhbjstpwgOS4364WE7JeQIOyCrOqzoCAz6rG07J6F64uI64ukLiAwIOyduCDshLjqt7jrqLztirgg7KSR"
           "IOqwgOyepSDsnpHsnYAgUjQyIMOXIEIxNCDripQg6rOE7JW9IDbqsbTCt+y0nSDrhbjstpwgMi40MuuFhCDrj5nslYgg7IKs6rOg"
           "6rCAIOyXhuyXiOydhCDrv5DsnoXri4jri6QuCi0g6rmU65WM6riwIO2Yle2DnOqwgCDsnbQg6rWs7KGw66W8IO2VnCDsnqXsl5Ag"
           "67O07JesIOykjeuLiOuLpCDigJQg7Jik66W47Kq9KOuFuOy2nOydtCDtgbAg7IS46re466i87Yq4KeyXkOyEnOuKlCDsoJDsnbQg"
           "7KCE7LK0IOqzhOyVvSDsgqzqs6DsnKgg6re87LKY7JeQIOuqqOydtOuKlOuNsCwg7Jm87Kq97Jy866GcIOqwiOyImOuhnSDshLjr"
           "oZzstpUg67Cp7ZalIOu2hOyCsOydtCDsu6Tsp5Hri4jri6QuIO2dqeyWtOynhCDsqr3snbQg6rSA7Lih7J20IOyggeydgCDsqr3s"
           "noXri4jri6QuIOy0nSDrhbjstpzsnbQgMTAw64WE7JeQIOuquyDrr7jsuZjripQg7IS46re466i87Yq46rCAIDI0MuqwnCDshLjq"
           "t7jrqLztirgg7KSRIDYz6rCc7J206rOgLCDrs4Drj5nsnbQg7YGwIOygkOydgCDqsbDsnZgg64ukIOq3uCDslYjsl5Ag7J6I7Iq1"
           "64uI64ukLgotIE1MRSDripQg7ZGc67O47JeQ7IScIOqzhOyCsOuQnCDqsJLsnYQg6re464yA66GcIOuwmO2ZmO2VmOuKlCDstpTs"
           "oJXrn4nsnbTrnbwsIOq0gOy4oeydtCA26rG07J2066m0IOq3uCDtkZzrs7gg67OA64+Z6rmM7KeAIOq3uOuMgOuhnCDspp3tj63t"
           "lanri4jri6QuICLsgqzqs6DsnKggMCIg7J2AIOychO2XmOydtCDsl4bri6TripQg65y77J20IOyVhOuLiOudvCDtjJDri6jtlaAg"
           "6rSA7Lih7J20IOu2gOyhse2VmOuLpOuKlCDrnLvsnbTqs6AsIOydtCDtkZzrpbwg6re464yA66GcIOqyrOyggSDsi5zsiqTthZzs"
           "l5Ag7KCB7Jqp7ZWY66m0IOq3uCDshLjqt7jrqLztirjsnZgg67O07ZeY66OM6rCAIDDsm5DsnbQg65Cp64uI64ukLiAyNDLqsJwg"
           "7IS46re466i87Yq466W8IOycoOyngO2VmOuptOyEnCDsnbQg67OA64+Z7J2EIOykhOydtOuKlCDrsKnrspXsnbQg64uk7J2MIOyg"
           "iOydmCDsgqzsoITrtoTtj6zsnoXri4jri6QuCgrsvZTrk5zqsIAg64u17J2EIOunjOuTnOuKlCDrsKnsi50g4oCUIOyEuOq3uOuo"
           "vO2KuOuniOuLpCDsgqzqs6Ag6rG07IiY7J2YIO2VqeydhCDqt7gg7IS46re466i87Yq47J2YIOuFuOy2nCDtlako64WEKeycvOuh"
           "nCDrgpjriKAg6re4IOyEuOq3uOuovO2KuOydmCDsl7DqsIQg7IKs6rOg7Jyo7J2EIOunjOuTreuLiOuLpCjsi53snYAg7KCE7LK0"
           "IOqzhOyVvSDsgqzqs6DsnKjqs7wg6rCZ6rOgLCDtlansnYQg64K064qUIOuylOychOunjCDshLjqt7jrqLztirgg7JWI7Jy866Gc"
           "IOyige2YlOyKteuLiOuLpCkuCg==",
           "m3":
                      "8J+UkyDrr7jshZggMyDsmYTshLHrs7gg4oCUIOOAjOyXrOq4sOu2gO2EsCBBSSDsvZTrk5zjgI0g7JWE656Y7JeQIOuTpOyWtOqw"
           "iCDsvZTrk5wg7JiI7Iuc7J6F64uI64ukCgrilqAg7J20IOy9lOuTnOulvCDrtoDrpbgg7ZSE66Gs7ZSE7Yq4IOyYiAoKICAgIOyd"
           "tOuvuCDrp4zrk6TslrTsoLgg7J6I64qUIHNlZ19maW5lLCB3YXRjaCwgbGFtX2FsbCwgYWxwaGFfMCwgYmV0YV8wIOydhCDqt7jr"
           "jIDroZwg7JOw6rOgIOuLpOyLnCDrp4zrk6Tsp4DripQg66eQ7JWEIOykmC4gc2VnX2ZpbmUg7J2AIOyEuOq3uOuovO2KuCDsnbTr"
           "poQKICAgIFJlZ2lvbiDCtyBWZWhCcmFuZCDsmYAg6rOE7JW9IOyImCBuIMK3IOyCrOqzoCDqsbTsiJggY2xhaW1zIMK3IOuFuOy2"
           "nCBleHBvc3VyZSDCtyDsgqzqs6DsnKggbGFtX21sZSDsl7TsnYQg6rCA7KeEIDI0MuqwnCDshLjqt7jrqLztirgg7ZGc7J206rOg"
           "LCB3YXRjaCDripQg6re47KSRCiAgICBSMjEgeCBCMTEg7IS46re466i87Yq4IO2VnCDtlonrp4wg7LaU7Lac7ZW0IOuRlCDtkZzs"
           "lbwuIHNlZ19maW5lIOyXkCDqsJDrp4gg7IKs7KCE67aE7Y+s66W8IOuwmOyYge2VnCDshLjqt7jrqLztirjrs4Qg7IKs6rOg7Jyo"
           "7J2EIGxhbV9tYXAg7Je066GcIOyDiOuhnCDrp4zrk6TslrQg7KSYIOKAlCDrtoTsnpDripQgYWxwaGFfMCDsl5Ag6re4IOyEuOq3"
           "uOuovO2KuOydmAogICAgY2xhaW1zIOulvCDrjZTtlZwg65KkIDEg7J2EIOu6gCDqsJIsIOu2hOuqqOuKlCBiZXRhXzAg7JeQIOq3"
           "uCDshLjqt7jrqLztirjsnZggZXhwb3N1cmUg66W8IOuNlO2VnCDqsJLsnbTslbwuIOq3uOumrOqzoCB3YXRjaCDtlZwg7ZaJ7JeQ"
           "IOqwmeydgCDsi53snYQg7KCB7Jqp7ZW0IOq3uCDshLjqt7jrqLztirjsnZggTUFQIOulvAogICAgIlIyMSB4IEIxMSDshLjqt7jr"
           "qLztirjsnZggTUFQID0gKOyIq+yekCkg6rG0L+uFhCIg7LKY65+8IOyGjOyImOygkCDslYTrnpggNuyekOumrOq5jOyngCDri6js"
           "nITsmYAg7ZWo6ruYIHByaW50IO2VmOqzoCDqsJnsnYAg7ZaJ7JeQIOq3uCDshLjqt7jrqLztirjsnZggbGFtX21sZSDrj4Qg6rSE"
           "7Zi466GcIO2VqOq7mCDsoIHslrQg7KSYLiAyNDLqsJwg7IS46re466i87Yq4CiAgICBsYW1fbWFwIOydmCDstZzshp/qsJLrj4Qg"
           "IjI0MuqwnCDshLjqt7jrqLztirggTUFQIOydmCDstZzshp/qsJIgPSAo7Iir7J6QKSDqsbQv64WEIiDsnLzroZwg7ZWcIO2WiSDr"
           "jZQgcHJpbnQg7ZWY6rOgIGxhbV9tbGUg7J20IDAg7J20642YIOyEuOq3uOuovO2KuOqwgCDrqocg6rCc7JiA64qU7KeAIO2VqOq7"
           "mCDstpzroKXtlbQg7KO86rOgLgogICAg6re4656Y7ZSE64qUIOuUsSDtlZwg7J6l66eMIOKAlCDsoJAg7ZWY64KY6rCAIOyEuOq3"
           "uOuovO2KuCDtlZjrgpjsnbgg7IKw7KCQ64+E66GcIOqwgOuhnOy2leyXkCDqt7gg7IS46re466i87Yq47J2YIGxhbV9tbGUsIOyE"
           "uOuhnOy2leyXkCBsYW1fbWFwIOydhCDrsLDsuZjtlZjqs6AsIOuRkCDqsJLsnbQg6rCZ7J2AIOyngOygkOydhCDsnofripQgeSA9"
           "IHgg64yA6rCB7ISg7J2EIDAKICAgIOu2gO2EsCBsYW1fbWxlIOydmCDstZzrjJPqsJLquYzsp4Ag7ZWY64KYIOq3uOydgCDri6Ts"
           "nYwg65GQIOy2leyXkCDrnbzrsqjsnYQg64us7JWEIOykmC4KCiAgICBzZWdfZmluZVsibGFtX21hcCJdID0gKGFscGhhXzAgKyBz"
           "ZWdfZmluZVsiY2xhaW1zIl0gLSAxKSAvIChiZXRhXzAgKyBzZWdfZmluZVsiZXhwb3N1cmUiXSkKCiAgICB3YXRjaF9sYW1fbWFw"
           "ID0gKGFscGhhXzAgKyB3YXRjaFsiY2xhaW1zIl0gLSAxKSAvIChiZXRhXzAgKyB3YXRjaFsiZXhwb3N1cmUiXSkgICAjIOOAjOyk"
           "gOu5hCDshYDjgI3snbQg7LaU7Lac7ZW0IOuRlCDtlZwg7ZaJCiAgICBwcmludChmIlIyMSB4IEIxMSDshLjqt7jrqLztirjsnZgg"
           "TUFQICA6IHtmbG9hdCh3YXRjaF9sYW1fbWFwLmlsb2NbMF0pOi42Zn0g6rG0L+uFhCIKICAgICAgICAgIGYiICAoTUxFIHtmbG9h"
           "dCh3YXRjaFsnbGFtX21sZSddLmlsb2NbMF0pOi42Zn0g6rG0L+uFhCkiKQogICAgcHJpbnQoZiIyNDLqsJwg7IS46re466i87Yq4"
           "IE1BUCDsnZgg7LWc7Iaf6rCSIDoge3NlZ19maW5lWydsYW1fbWFwJ10ubWluKCk6LjZmfSDqsbQv64WEIgogICAgICAgICAgZiIg"
           "IHwgTUxFIOqwgCAwIOydtOuNmCDshLjqt7jrqLztirgge2ludCgoc2VnX2ZpbmVbJ2xhbV9tbGUnXSA9PSAwKS5zdW0oKSl9IOqw"
           "nCIpCgogICAgYXhpc19lbmQgPSBmbG9hdChzZWdfZmluZVsibGFtX21sZSJdLm1heCgpKSAgICAgICAgICAgICAgICMg64yA6rCB"
           "7ISg7J2EIOq3uOydhCDqtazqsITsnZgg7Jik66W47Kq9IOuBnQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg4"
           "LCA0KSkKICAgIGF4LnNjYXR0ZXIoc2VnX2ZpbmVbImxhbV9tbGUiXSwgc2VnX2ZpbmVbImxhbV9tYXAiXSwgcz0xNCkKICAgIGF4"
           "LnBsb3QoWzAsIGF4aXNfZW5kXSwgWzAsIGF4aXNfZW5kXSwgbHc9MSwgY29sb3I9ImdyYXkiKSAgICAgICAjIOuRkCDstpTsoJXq"
           "sJLsnbQg6rCZ7J2AIOyngOygkCDigJQg64yA6rCB7ISgCiAgICBheC5zZXRfeGxhYmVsKCJNTEUg4oCUIOuNsOydtO2EsOqwgCDr"
           "p5DtlZwg6re464yA66GcICjqsbQv64WEKSIpCiAgICBheC5zZXRfeWxhYmVsKCJNQVAg4oCUIOqwkOuniCDsgqzsoITrtoTtj6zr"
           "pbwg67CY7JiB7ZWcIOuSpCAo6rG0L+uFhCkiKQogICAgcGx0LnNob3coKQoK7ZW07ISdCgotIOy9lOuTnCDtlZwg7ZaJ7J20IDI0"
           "MuqwnCDshLjqt7jrqLztirjrpbwg64ukIOqzhOyCsO2VqeuLiOuLpC4g7YyQ64uk7IqkIOyXtOuBvOumrOydmCDrjafshYjCt+uC"
           "mOuIl+yFiOydtOudvCDrsJjrs7XrrLjsnbQg7ZWE7JqUIOyXhuyKteuLiOuLpC4g7Zi864+Z65CY64qUIOu2gOu2hOydgCAtMSDs"
           "nYQg7Ja065SU7IScIOu5vOuKlOqwgCDtlZjrgpjsnoXri4jri6QuIOyCrO2bhOu2hO2PrOydmCDrqqjsiJjsl5DshJzqsIAg7JWE"
           "64uI6528IOy1nOu5iOqwkuydhCDqtaztlZjripQg66eI7KeA66eJIOuLqOqzhOyXkOyEnCAx7ZqM66eMIOu6jeuLiOuLpC4g65GQ"
           "IOuyiCDrubzrqbQg66qo65OgIOqwkuydtCAxLyhiZXRhXzAgKyDsi6TsoJwg64W47LacKSDrp4ztgbwg64Ku7JWE7KeR64uI64uk"
           "LiDrhbjstpwgNi4264WE7J2066m0IDAuMDA5NCwg6rOnIDglIOqwkOyGjO2VqeuLiOuLpC4KLSBSMjEgeCBCMTEg7IS46re466i8"
           "7Yq464qUIDAuMzAyMTE1IC0+IDAuMTEzMTI1IOuhnCDqsJDshoztlojsirXri4jri6QuIOu2hOyekOuKlCDqsIDsg4Eg7IKs6rOg"
           "IDEwLjA26rG07JeQIOyLpOygnCDsgqzqs6AgMuqxtOydhCDrjZTtlZwgMTIuMDYg7J6F64uI64ukLiDrtoTrqqjripQg6rCA7IOB"
           "IOuFuOy2nCAxMDDrhYTsl5Ag7Iuk7KCcIOuFuOy2nCA2LjYy64WE7J2EIOuNlO2VnCAxMDYuNjIg7J6F64uI64ukLiDqt7jspJEg"
           "7J20IOyEuOq3uOuovO2KuOydmCDrjbDsnbTthLAg6rCA7KSR7LmYIHcg64qUIDYuMiUg67+Q7J6F64uI64ukLiBNTEUgMC4zMDIg"
           "66W8IOuSt+uwm+y5qO2VmOuKlCDqt7zqsbDqsIAg6rGw7J2YIOyXhuyKteuLiOuLpC4KLSDsgqzqs6DsnKjsnbQgMC4wMDAwMDAg"
           "7J20642YIOyEuOq3uOuovO2KuCA56rCc64qUIDAuMDg0OTIwIH4gMC4wOTgyMzcg66GcIOymneqwgO2WiOyKteuLiOuLpC4gMjQy"
           "6rCcIE1BUCDsnZgg7LWc7Iaf6rCS7J2AIDAuMDU1MDYzIOqxtC/rhYTsnbTrnbwgMCDsnbgg7IS46re466i87Yq46rCAIOyXhuyK"
           "teuLiOuLpC4g67O07ZeY66OMIDDsm5Ag66y47KCc6rCAIO2VtOyGjOuQqeuLiOuLpC4g64uk66eMIOq3uCDstZzshp/qsJLsnbQg"
           "64KY7JioIOqzs+ydgCDrrLTsgqzqs6Ag7IS46re466i87Yq46rCAIOyVhOuLmeuLiOuLpC4g64W47LacIDEsNTU064WE7Kec66as"
           "IOyggOychO2XmCDrjIDtmJUg7IS46re466i87Yq4IFI0MSB4IEIxKE1MRSAwLjA1MjEzMinsnbTrnbwsIOybkOuemCDrgq7snYAg"
           "6rCS7J20IOycoOyngOuQnCDqsr3smrDsnoXri4jri6QuCi0g7JiI7IOBIDPqsJwg7KSRIOuLteydgCAy67KILCDjgIzshoztmJUg"
           "7IS46re466i87Yq467O064ukIO2bqOyUrCDsnpHqsowsIOydtOuPmSDqsbDrpqzqsIAgMC4wMDEg7JeQ64+EIOuquyDrr7jsuZzr"
           "i6TjgI0g7J6F64uI64ukLiDrhbjstpwgMzUsOTQ564WE7Kec66asIOy1nOuMgCDshLjqt7jrqLztirggUjI0IHggQjIg64qUIDAu"
           "MDg4OTg4IC0+IDAuMDg5MDIwLCDsnbTrj5kg6rGw66asIDAuMDAwMDMyIOyeheuLiOuLpC4g64W47LacIDHrp4wg64WE7J2EIOuE"
           "mOuKlCA26rCc64+EIOy1nOuMgCAwLjAwMDM5IOyeheuLiOuLpC4gM+uyiCDjgIzslYTsmIgg7JuA7KeB7J207KeAIOyViuuKlOuL"
           "pOOAjSDqsIAg64u17J20IOyVhOuLjCDsnbTsnKDrj4Qg6rCZ7Iq164uI64ukIOKAlCDsnbTrj5kg6rGw66as6rCAIOyekeydhCDr"
           "v5AgMCDsnYAg7JWE64uZ64uI64ukLiDsgrDsoJDrj4Tsl5DshJwg64yA6rCB7ISg7JeQIOyduOygke2VnCDsoJDrk6TsnbQg7J20"
           "IOynkeuLqOyeheuLiOuLpC4KLSDqtIDsuKHsnbQg7KCB7J2A7KeA66W8IO2MkOuLqO2VmOuKlCDquLDspIDsnYAg6rOE7JW9IOyI"
           "mOqwgCDslYTri4jrnbwg64W47Lac7J6F64uI64ukLiDrjIDqsIHshKDsl5DshJwg6rCA7J6lIOupgOumrCDrlqjslrTsp4QgUjQz"
           "IHggQjQg64qUIOqzhOyVvSAyNuqxtOyXkCDrhbjstpwgOS4364WE7J206rOgLCAwLjMwOTE5MSAtPiAwLjExOTA2MiDsnLzroZwg"
           "7J2064+Z7ZWp64uI64ukLiDqs4Tslb3snbQgMjIz6rG07J24IFIyMSB4IEIzIOuPhCDstJ0g64W47Lac7J2AIDg4LjA164WE67+Q"
           "7J206528IDAuMjk1MjgwIC0+IDAuMTkxNzYzIOycvOuhnCDtgazqsowg7J2064+Z7ZWp64uI64ukLiDrhbjstpwgMTAw64WEIOuv"
           "uOunjCA2M+qwnOydmCDtj4nqt6Ag7J2064+ZIOqxsOumrOuKlCAwLjA0NjIsIDEsMDAw64WEIOydtOyDgSA3NeqwnOuKlCAwLjAw"
           "MDgg7J6F64uI64ukLiA1OeuwsCDssKjsnbTqsIAg44CM6rSA7Lih7J20IOyggeydgCDshLjqt7jrqLztirjsl5Drp4wg7YGs6rKM"
           "IOyggeyaqeuQnOuLpOOAjeulvCDsiKvsnpDroZwg7KCB7J2AIOqyg+yeheuLiOuLpC4KCuy9lOuTnOqwgCDri7XsnYQg66eM65Oc"
           "64qUIOuwqeyLnSDigJQg7L2U65Oc64qUIOyEuOq3uOuovO2KuOuniOuLpCDrtoTsnpDsl5Ag7IKs7KCE67aE7Y+s7J2YIOqwgOyD"
           "gSDsgqzqs6AoYWxwaGFfMCAtIDEp66W8LCDrtoTrqqjsl5Ag6rCA7IOBIOuFuOy2nCBiZXRhXzAg66W8IOuNlO2VnCDrkqQg64KY"
           "64iV64uI64ukLiDsi6TsoJwg64W47Lac7J20IO2BtOyImOuhnSDrtoTrqqjsl5DshJwg7IKs7KCE67aE7Y+s7J2YIOu5hOykkeyd"
           "tCDsnpHslYTsoLgg6re4IOyEuOq3uOuovO2KuOuKlCDqsbDsnZgg7J2064+Z7ZWY7KeAIOyViuyKteuLiOuLpC4g7Iuk7KCcIOuF"
           "uOy2nOydtCDsnpHsnYTsiJjroZ0g7IKs7KCE67aE7Y+s6rCAIOu2hOuqqOulvCDssKjsp4DtlbQg7LWc67mI6rCS7J20IOyghOyy"
           "tCDqs4Tslb3snZgg7Jew6rCEIOyCrOqzoOycqCDsqr3snLzroZwg7J2064+Z7ZWp64uI64ukLgo=",
           "과제":
           "8J+UkyDqs7zsoJwg7JmE7ISx67O4IOKAlCDjgIzsl6zquLDrtoDthLAgQUkg7L2U65Oc44CNIOyVhOuemOyXkCDrk6TslrTqsIgg7L"
           "2U65OcIOyYiOyLnOyeheuLiOuLpAoK4pagIOydtCDsvZTrk5zrpbwg67aA66W4IO2UhOuhrO2UhO2KuCDsmIgKCiAgICDsnbTrr7gg"
           "66eM65Ok7Ja07KC4IOyeiOuKlCBzZWdfYmluLCBwX2dsb2JhbCwgbnUsIHJlZ19iaW4sIHJuZyDsnYQg6re464yA66GcIOyTsOqzoC"
           "Dri6Tsi5wg66eM65Ok7KeA64qUIOunkOyVhCDspJguIOy1nOu5iOqwkuydhCDsoITssrQg6rOE7JW97J2YIOyCrOqzoCDqsr3tl5jr"
           "paAKICAgIHBfZ2xvYmFsIOyXkCDrp57stpTqs6Ag6rCA7IOBIOqzhOyVvSDsiJjrpbwgbnUg6rG07Jy866GcIOyEpOygle2VnCDrsq"
           "Dtg4Ag7IKs7KCE67aE7Y+s66W8IOygle2VtCDspJgg4oCUIOyyqyDrqqjsiJjripQgMSArIHBfZ2xvYmFsIHggbnUsIOuRmOynuCDr"
           "qqjsiJjripQKICAgIDEgKyAoMSAtIHBfZ2xvYmFsKSB4IG51IOyVvC4g6re4IOyCrOyghOycvOuhnCAyNDLqsJwg7IS46re466i87Y"
           "q4IO2RnCBzZWdfYmluIOyXkCDshLjqt7jrqLztirjrs4Qg7IKs6rOgIOqyve2XmOuloOydmCBNQVAg66W8IHBfbWFwIOyXtOuhnCDr"
           "p4zrk6TslrQg7KSYLiDrtoTsnpDripQg7LKrCiAgICDrqqjsiJjsl5Ag6re4IOyEuOq3uOuovO2KuOydmCDsgqzqs6Ag6rOE7JW9IO"
           "yImCBzIOulvCDrjZTtlZwg65KkIDEg7J2EIOu5vOqzoCwg67aE66qo64qUIOuRkCDrqqjsiJjsnZgg7ZWp7JeQIOq3uCDshLjqt7jr"
           "qLztirjsnZgg6rOE7JW9IOyImCBuIOydhCDrjZTtlZwg65KkIDIg66W8IOu6gCDqsJLsnbTslbwuIOq3uOufsCDri6TsnYwKICAgIC"
           "LsnbTrpoQgPSAo7Iir7J6QKSDri6jsnIQiIO2Yle2DnOuhnCDqsIHqsIEgcHJpbnQg7ZW0IOykmCDigJQg66y07IKs6rOg7Jyo7J20"
           "IOuGkuydgCDsiJwg7KaJIHBfbWxlIOydtCDsnpHsnYAg7Iic7Jy866GcIOqzoOuluCDsl7Qg7IS46re466i87Yq47J2YIOqzhOyVvS"
           "DsiJggbiDtlako6rG0KSwg6rCZ7J2ACiAgICDrsKnsi53snLzroZwgcF9tYXAg7J20IOyekeydgCDsiJzsnLzroZwg6rOg66W4IOyX"
           "tCDshLjqt7jrqLztirjsnZgg6rOE7JW9IOyImCBuIO2VqSjqsbQpLCDqt7jrpqzqs6AgMSDsl5DshJwgcF9nbG9iYWwg7J2EIOu6gC"
           "DrrLTsgqzqs6Ag67mE7Jyo7JeQIHNlZ19iaW4g7J2YIOqzhOyVvSDsiJggbgogICAg7LSd7ZWp7J2EIOqzse2VtCDrsJjsmKzrprzt"
           "lZwg66y07IKs6rOgIO2ZmOq4iSDrjIDsg4Eg6rOE7JW9IOyImCjqsbQpLgogICAgCiAgICDqt7jrpqzqs6AgcmVnX2JpbiDsl5DshJ"
           "wgUmVnaW9uIOydtCBSNDMg7J24IO2WieqzvCBSNDIg7J24IO2WieydhCDqs6jrnbwg6rCZ7J2AIOyCrOyghOu2hO2PrOuhnCDsgqzt"
           "m4TrtoTtj6zrpbwg7ZWY64KY7JSpIOunjOuTpOyWtCDspJgg4oCUIOyyqyDrqqjsiJjripQg7IKs7KCEIOyyqyDrqqjsiJjsl5AKIC"
           "AgIOq3uCDsp4Dsl63snZgg7IKs6rOgIOqzhOyVvSDsiJggcyDrpbwg642U7ZWcIOqwkiwg65GY7Ke4IOuqqOyImOuKlCDsgqzsoIQg"
           "65GY7Ke4IOuqqOyImOyXkCDqs4Tslb0g7IiYIG4g7JeQ7IScIHMg66W8IOu6gCDqsJLsnYQg642U7ZWcIOqxsOyVvC4g65GQIOyCrO"
           "2bhOu2hO2PrOyXkOyEnCBybmcg66GcCiAgICDtkZzrs7ggMTDrp4wg6rCc7JSpIOy2lOy2nO2VtCBSNDIg7Kq97J20IOuNlCDtgbAg"
           "7ZGc67O47J2YIOu5hOycqOydhCAiUjQyIOqwgCBSNDMg67O064ukIOuGkuydhCDtmZXrpaAgPSAo7Iir7J6QKSDtmZXrpaAiIOuhnC"
           "BwcmludCDtlZjqs6AsIOuRkCDtkZzrs7jsnZgg7LCo7J2066GcIDk1JSDqtazqsITrj4QKICAgIO2VnCDtlokg7LaU6rCA7ZW0IOyk"
           "mC4KICAgIAogICAg64yA7KGw7Jqp7Jy866GcIFJlZ2lvbiDsnbQgUjIxIOydtOqzoCBWZWhCcmFuZCDqsIAgQjExIOyduCDshLjqt7"
           "jrqLztirjsnZggcF9tYXAo67mE7JyoLCDshozsiJjsoJAg7JWE656YIDbsnpDrpqwp6rO8IDI0MuqwnCDshLjqt7jrqLztirggcF9t"
           "YXAg7J2YIOy1nOyGn+qwkuuPhCDtlZwg7ZaJ7JSpCiAgICDstpzroKXtlbQg7KO86rOgLiDqt7jrnpjtlITripQg65SxIO2VnCDsnq"
           "Xrp4wg4oCUIOygkCDtlZjrgpjqsIAg7IS46re466i87Yq4IO2VmOuCmOyduCDsgrDsoJDrj4TroZwg6rCA66Gc7LaV7JeQIHBfbWxl"
           "LCDshLjroZzstpXsl5AgcF9tYXAg7J2EIOuwsOy5mO2VmOqzoCDrkZAg6rCS7J20IOqwmeydgCDsp4DsoJDsnYQg7J6H64qUCiAgIC"
           "B5ID0geCDrjIDqsIHshKDsnYQg7ZWY64KYIOq3uOydgCDrkqQg65GQIOy2leyXkCDrnbzrsqjsnYQg64us7JWEIOykmC4KCiAgICBh"
           "bHBoYV9iID0gMSArIHBfZ2xvYmFsICogbnUgICAgICAgICAgICAgICAgICAjIOy1nOu5iOqwkuydhCDsoITssrQg6rOE7JW97J2YIO"
           "yCrOqzoCDqsr3tl5jrpaDsl5Ag66ee7LaY64ukCiAgICBiZXRhX2IgPSAxICsgKDEgLSBwX2dsb2JhbCkgKiBudSAgICAgICAgICAg"
           "ICAjIOuRkCDrqqjsiJjsnZgg7ZWp7JeQ7IScIDIg66W8IOu5vOuptCDqsIDsg4Eg6rOE7JW9IOyImCBudSDqsIAg65Cp64uI64ukCi"
           "AgICBzZWdfYmluWyJwX21hcCJdID0gKGFscGhhX2IgKyBzZWdfYmluWyJzIl0gLSAxKSAvIChhbHBoYV9iICsgYmV0YV9iICsgc2Vn"
           "X2JpblsibiJdIC0gMikKCiAgICB0b3BfbWxlID0gc2VnX2Jpbi5uc21hbGxlc3QoMTAsICJwX21sZSIpICAgICAjIOustOyCrOqzoO"
           "ycqCDsg4HsnIQgMTDqsJwgPSDsgqzqs6Ag6rK97ZeY66WgIO2VmOychCAxMOqwnAogICAgdG9wX21hcCA9IHNlZ19iaW4ubnNtYWxs"
           "ZXN0KDEwLCAicF9tYXAiKQogICAgcHJpbnQoZiJNTEUg66GcIOqzoOuluCDsg4HsnIQg7Je0IOyEuOq3uOuovO2KuOydmCDqs4Tslb"
           "0g7IiYID0ge2ludCh0b3BfbWxlWyduJ10uc3VtKCkpOix9IOqxtCIpCiAgICBwcmludChmIk1BUCDroZwg6rOg66W4IOyDgeychCDs"
           "l7Qg7IS46re466i87Yq47J2YIOqzhOyVvSDsiJggPSB7aW50KHRvcF9tYXBbJ24nXS5zdW0oKSk6LH0g6rG0IikKICAgIHByaW50KG"
           "Yi64yA7IOBIOqzhOyVvSDsiJggPSB7aW50KHJvdW5kKCgxIC0gcF9nbG9iYWwpICogc2VnX2JpblsnbiddLnN1bSgpKSk6LH0g6rG0"
           "IikKCiAgICByZWdpb25fcm93cyA9IHtyOiByZWdfYmluW3JlZ19iaW5bIlJlZ2lvbiJdID09IHJdLmlsb2NbMF0gZm9yIHIgaW4gKC"
           "JSNDMiLCAiUjQyIil9CiAgICBkcmF3cyA9IHtyOiBybmcuYmV0YShhbHBoYV9iICsgcmVnaW9uX3Jvd3Nbcl1bInMiXSwgYmV0YV9i"
           "ICsgcmVnaW9uX3Jvd3Nbcl1bIm4iXSAtIHJlZ2lvbl9yb3dzW3JdWyJzIl0sIDEwMF8wMDApCiAgICAgICAgICAgIGZvciByIGluIC"
           "giUjQzIiwgIlI0MiIpfSAgICAgICAgICAgICAgICAjIOy2lOy2nCDsiJzshJzrj4Qg64Kc7IiYIOyLnOuTnOydmCDsnbzrtoDsnoXr"
           "i4jri6Qg4oCUIFI0MyDrtoDthLAKICAgIHJlZ2lvbl9nYXAgPSBkcmF3c1siUjQyIl0gLSBkcmF3c1siUjQzIl0KICAgIHByaW50KG"
           "YiUjQyIOqwgCBSNDMg67O064ukIOuGkuydhCDtmZXrpaAgPSB7KGRyYXdzWydSNDInXSA+IGRyYXdzWydSNDMnXSkubWVhbigpOi40"
           "Zn0gKO2ZleuloCkiKQogICAgcHJpbnQoZiLssKjsnbTsnZggOTUlIOyLoOyaqeq1rOqwhCA9IHtucC5wZXJjZW50aWxlKHJlZ2lvbl"
           "9nYXAsIDIuNSk6Ky40Zn0gfiB7bnAucGVyY2VudGlsZShyZWdpb25fZ2FwLCA5Ny41KTorLjRmfSAo67mE7JyoKSIpCgogICAgc2Vn"
           "X3JvdyA9IHNlZ19iaW5bKHNlZ19iaW5bIlJlZ2lvbiJdID09ICJSMjEiKSAmIChzZWdfYmluWyJWZWhCcmFuZCJdID09ICJCMTEiKV"
           "0uaWxvY1swXQogICAgcHJpbnQoZiIo64yA7KGwKSDsgqzsoITrtoTtj6wgQmV0YSh7YWxwaGFfYjouMmZ9LCB7YmV0YV9iOi4yZn0p"
           "IgogICAgICAgICAgZiIgfCBSMjEgeCBCMTEg7IS46re466i87Yq47J2YIE1BUCA9IHtzZWdfcm93WydwX21hcCddOi42Zn0gKOu5hO"
           "ycqCkiKQogICAgcHJpbnQoZiIo64yA7KGwKSAyNDLqsJwg7IS46re466i87Yq4IE1BUCDsnZgg7LWc7Iaf6rCSID0ge3NlZ19iaW5b"
           "J3BfbWFwJ10ubWluKCk6LjZmfSAo67mE7JyoKSIpCgogICAgYXhpc19lbmQgPSBmbG9hdChzZWdfYmluWyJwX21sZSJdLm1heCgpKS"
           "AgICAgICAgICAgICAgICAgICAgICAgIyDrjIDqsIHshKDsnYQg6re47J2EIOq1rOqwhOydmCDsmKTrpbjsqr0g64GdCiAgICBmaWcs"
           "IGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDgsIDQuMikpCiAgICBheC5zY2F0dGVyKHNlZ19iaW5bInBfbWxlIl0sIHNlZ19iaW"
           "5bInBfbWFwIl0sIHM9MTIpCiAgICBheC5wbG90KFswLCBheGlzX2VuZF0sIFswLCBheGlzX2VuZF0sIGx3PTEsIGNvbG9yPSJncmF5"
           "IikgICAgICAjIOuRkCDstpTsoJXqsJLsnbQg6rCZ7J2AIOyngOygkCDigJQg64yA6rCB7ISgCiAgICBheC5zZXRfeGxhYmVsKCJNTE"
           "Ug4oCUIOuNsOydtO2EsOqwgCDrp5DtlZwg6re464yA66GcICjruYTsnKgpIikKICAgIGF4LnNldF95bGFiZWwoIk1BUCDigJQg67Kg"
           "7YOAIOyCrOyghOu2hO2PrOulvCDrsJjsmIHtlZwg65KkICjruYTsnKgpIikKICAgIHBsdC5zaG93KCkKCuKWoCDtkZzrs7jstpTstp"
           "wg64yA7IugIOqyqeyekCDsoIHrtoTsnLzroZwg4oCUIOuzgOuPme2VmOyngCDslYrripQg7KCV67O4IOqwkgoKICAgIGdyaWQgPSBu"
           "cC5saW5zcGFjZSgwLCAxLCA1MF8wMDApCiAgICBwb3N0ZXJpb3IgPSB7cjogKGFscGhhX2IgKyByZWdpb25fcm93c1tyXVsicyJdLC"
           "BiZXRhX2IgKyByZWdpb25fcm93c1tyXVsibiJdIC0gcmVnaW9uX3Jvd3Nbcl1bInMiXSkgZm9yIHIgaW4gKCJSNDMiLCAiUjQyIil9"
           "CiAgICBpbnRlZ3JhbmQgPSBzdGF0cy5iZXRhLnBkZihncmlkLCAqcG9zdGVyaW9yWyJSNDIiXSkgKiBzdGF0cy5iZXRhLmNkZihncm"
           "lkLCAqcG9zdGVyaW9yWyJSNDMiXSkKICAgIHByaW50KGYi6rKp7J6Q66GcIOyggeu2hO2VnCDtmZXrpaAgPSB7ZmxvYXQoaW50ZWdy"
           "YW5kLnN1bSgpICogKGdyaWRbMV0gLSBncmlkWzBdKSk6LjRmfSAo7ZmV66WgKSIpCgogIO2RnOuzuCAxMOunjCDqsJzroZwg6rOE7I"
           "Kw7ZWcIDAuOTc2NSDsmYAg6rKp7J6QIOyggeu2hOqwkiAwLjk3Njcg7J2AIOqwmeydgCDqsJLsnZgg7ZGc6riwIDLqsIDsp4DsnoXr"
           "i4jri6QuIO2RnOuzuOy2lOy2nOydgCDrgpzsiJgg7Iuc65Oc7JmACiAg7ZGc67O4IOyImOyXkCDrlLDrnbwg66eI7KeA66eJIOyekO"
           "umrOqwgCDrs4Drj5ntlZjqs6Ag6rKp7J6QIOyggeu2hOydgCDrs4Drj5ntlZjsp4Ag7JWK7Iq164uI64ukIOKAlCDrs7Tqs6DshJzs"
           "l5Ag7KCB7J2EIOqwkuydgCDqsqnsnpAg7KCB67aE7Jy866GcCiAg7ZWcIOuyiCDrp57strAg67O06rOgIOyUgeuLiOuLpC4g7LGE7K"
           "CQIOuylOychCAwLjk2NSB+IDAuOTkwIOydgCDqt7gg67OA64+ZIOuylOychOydmCAz67Cw66GcIOyEpOygle2VtCDrkZDsl4jsirXr"
           "i4jri6QuCgrsnb3ripQg67KVCgogIMK3IOyCrOyghOu2hO2PrOuKlCBCZXRhKDYuMDIsIDk1Ljk4KSDsnoXri4jri6Qg4oCUIOqwgO"
           "yDgSDqs4Tslb0gMTAw6rG0IOykkSDsgqzqs6Ag6rOE7JW9IOyVvSA16rG07J2EIOuvuOumrCDqtIDsuKHtlojri6Tqs6Ag6rCA7KCV"
           "7ZWY64qUIOqyg+qzvCDqsJnsirXri4jri6QuCiAgwrcg66eI7LyA7YyF7YyA7J20IOusvOyWtOuzuCDrjIDtkZwg7Iir7J6QLCDrrL"
           "Tsgqzqs6Ag7ZmY6riJIOuMgOyDgSDqs4Tslb0g7IiY64qUIDY0Myw5NTPqsbTsnoXri4jri6QKICAgICjsoITssrQgNjc4LDAxM+qx"
           "tOydmCA5NS4wJSkuIOydtCDtlZwg66y47J6l7J2AIOyghOyytCDqs4Tslb3snZgg7IKs6rOgIOqyve2XmOuloOunjOycvOuhnOuPhC"
           "DrgpjsmKTqs6AsIDI0MuqwnCDshLjqt7jrqLztirggTUFQIOuhnCDqsIDspJHtlbQKICAgIOuNlO2VtOuPhCA2NDMsODgz6rG07J20"
           "6528IDcw6rG067CW7JeQIOuLrOudvOyngOyngCDslYrsirXri4jri6Qg4oCUIOq3nOuqqOunjCDrs7TrqbQg7Jik64qYIOuwsOyatC"
           "DsgqzsoITrtoTtj6zqsIAg7JeG7Ja064+EIOqwmeyKteuLiOuLpC4KICDCtyDri6zrnbzsp4DripQg6rKD7J2AIOq3nOuqqOqwgCDs"
           "lYTri4jrnbwg66qF64uo7J6F64uI64ukLiBNTEUg66GcIOustOyCrOqzoOycqCDsg4HsnIQg7IS46re466i87Yq4IDEw6rCc66W8IO"
           "qzoOultOuptCDqs4Tslb0gNn45MeqxtOyduAogICAg7IaM7ZiVIOyEuOq3uOuovO2KuOunjCDshKDtg53rj7wgMjM56rG07J206rOg"
           "LCBNQVAg66GcIOqzoOultOuptCDqs4Tslb0gMjQxfjIsNTc56rG07J24IOyEuOq3uOuovO2KuOqwgCDshKDtg53rj7wgOCw4Nznqsb"
           "TsnoXri4jri6QuCiAgICDrkZAg66qF64uo7J2AIO2VnCDqsJzrj4Qg6rK57LmY7KeAIOyViuyKteuLiOuLpCDigJQgTUxFIOuKlCAi"
           "7IKs6rOg66W8IOyViCDrgrTripQg7IS46re466i87Yq4IiDqsIAg7JWE64uI6528ICLslYTsp4Eg7JWE66y0IOydvOuPhAogICAg7J"
           "287Ja064KY7KeAIOyViuydgCDsnpHsnYAg7IS46re466i87Yq4IiDrpbwg7ISg7YOd7ZWY6riwIOuVjOusuOyeheuLiOuLpC4g66qF"
           "64uo7J2EIE1MRSDroZwg7KCV7ZWY66m0IO2ZmOq4iSDrjIDsg4HsnbQg7ZGc67O4IOuzgOuPmeyXkCDrlLDrnbwg6rKw7KCV65Cp64"
           "uI64ukLgogIMK3IOuRkCDsp4Dsl63snZgg7IKs7ZuE67aE7Y+s64qUIEJldGEoNjAuMDIsIDEzNjguMCkg6rO8IEJldGEoMTMwLjAy"
           "LCAyMTcyLjApIOydtOqzoCwg7IKs7ZuE67aE7Y+s7J2YIO2Pieq3oOydgCAwLjA0MjAg6rO8CiAgICAwLjA1NjUg7J6F64uI64ukLi"
           "DtmZXrpaAgMC45NzY1IOuKlCDssKjsnbTsnZgg67Cp7Zal66eMIOuCmO2DgOuDheuLiOuLpCDigJQg7LCo7J207J2YIOq1rOqwhOyd"
           "gCDqs4Tslb0gMeunjCDqsbTri7kgMuqxtOyXkOyEnCAyODTqsbTquYzsp4DrnbwKICAgIOyDge2VnOydtCDtlZjtlZzsnZggMTDrsL"
           "DqsIAg64SY7Iq164uI64ukLiDrsKntlqXsnbQg7KCV7ZW07KeEIOqyg+qzvCDssKjsnbTqsIAg7YGwIOqyg+ydgCDshJzroZwg64uk"
           "66W4IOynhOyIoOyeheuLiOuLpC4KICDCtyDsnbTroaAgNeygiOydgCDqt6Drk7Eg7IKs7KCE67aE7Y+sIEJldGEoMSwgMSkg66GcIO"
           "qwmeydgCDruYTqtZDrpbwg7IiY7ZaJ7ZW0IDAuOTgwIMK3IDHrp4wg6rG064u5IDfvvZ4yOTbqsbTsnYQg7IKw7Lac7ZaI7Iq164uI"
           "64ukLgogICAg7Jes6riw7ISc64qUIEJldGEoNi4wMiwgOTUuOTgpIOydhCDsjaggMC45NzY1IMK3IDLvvZ4yODTqsbTsnbQg65Cp64"
           "uI64ukIOKAlCDssKjsnbTrpbwg66eM65OgIOqyg+ydgCDsgqzsoITrtoTtj6zsnoXri4jri6QuCiAgwrcg6rCZ7J2AIOuRkCDsp4Ds"
           "l63snYQg7IKs6rOgIOqxtOyImCDDtyDrhbjstpwg7KeA7ZGc66GcIOuLpOyLnCDsuKHsoJXtlZjrqbQg7J20IO2ZleuloOydtCAwLj"
           "cxIOuhnCDqsJDshoztlanri4jri6Qo7J2066GgIDXsoIgpLgogICAgUjQyIOydmCDrhbjstpzsnbQg7JW9IDMwJSDrjZQg6ri47Ja0"
           "7IScIOyDneq4sOuKlCDssKjsnbTsnbTrr4DroZwsIOyalOycqCDtmozsnZjsl5DshJzripQg7Lih7KCVIOyngO2RnOulvCDrqLzsoI"
           "Ag7ZmV7KCV7ZW07JW8IO2VqeuLiOuLpC4KICDCtyDrjIDsobAg7ZaJ64+EIO2VqOq7mCDrs7TshLjsmpQuIOqzhOyVvSAxMuqxtCDs"
           "pJEg7IKs6rOgIDLqsbTsnbTrjZggUjIxIHggQjExIOyEuOq3uOuovO2KuOydmCBNTEUgMC4xNjY3IOydgCBNQVAgMC4wNjI3IOuhnA"
           "ogICAg6rCQ7IaM7ZWp64uI64ukIOKAlCDsoITssrQg6rOE7JW9IOyCrOqzoCDqsr3tl5jrpaDsnZggMy4z67CwIO2VoOymneydtCAx"
           "LjI167Cw66GcIOqwkOyGjO2WiOyKteuLiOuLpC4g7IKs6rOg6rCAIO2VnCDqsbTrj4Qg7JeG642YCiAgICDshLjqt7jrqLztirggOe"
           "qwnOydmCAwLjAwMDAg7J2AIDAuMDM3NSB+IDAuMDQ3NCDroZwg7Kad6rCA7ZWY6rOgLCAyNDLqsJwg7IS46re466i87Yq4IE1BUCDs"
           "nZgg7LWc7Iaf6rCS7J2AIDAuMDIzOTUxIOydtOudvAogICAgMjQy6rCcIOyEuOq3uOuovO2KuCDslrTrlJTsl5Drj4QgMCDsnbQg64"
           "Ko7KeAIOyViuyKteuLiOuLpCjruIzrpqztlZHsnZgg7JyE7ZeYIO2VreuqqSDqt7zqsbApLgogIMK3IOqzhOyVveydtCA166eMIOqx"
           "tOydhCDrhJjripQg7LWc64yAIOyEuOq3uOuovO2KuOuKlCDshozsiJjsoJAg7JWE656YIDTsnpDrpqzquYzsp4AgTUxFIOyZgCDqsJ"
           "nsirXri4jri6QuIOuMgOqwgeyEoCDsnITsl5Ag6re464yA66GcIOychOy5mO2VqeuLiOuLpC4KICDCtyDrr7jshZggMyDsnZgg7IKw"
           "7KCQ64+E7JmAIOuqqOyWkeydtCDqsJnsirXri4jri6QuIOu2hO2PrCDqsIDsobHrp4wg6rCQ66eI7JeQ7IScIOuyoO2DgOuhnCDrsJ"
           "Tqv6jsnYQg67+QIOygiOywqOuKlCDrj5nsnbztlanri4jri6QKICAgIOKAlCDsvKTroIgg7IKs7KCE67aE7Y+s7J206riw66eMIO2V"
           "mOuptCDqsJnsnYAg67Kg7J207KaIIOygiOywqOqwgCDqt7jrjIDroZwg7KCB7Jqp65Cp64uI64ukLgoK7L2U65Oc6rCAIOuLteydhC"
           "Drp4zrk5zripQg67Cp7IudIOKAlCDsgqzsoITrtoTtj6zsnZgg6rCA7IOBIOqzhOyVvSBudSDqsbTsnYQg67aE66qo7JeQLCDqsIDs"
           "g4Eg7IKs6rOgIOqzhOyVvSAo7KCE7LK0IOqzhOyVveydmCDsgqzqs6Ag6rK97ZeY66WgIHggbnUpIOqxtOydhArrtoTsnpDsl5Ag64"
           "2U7ZW0IOuRkOyXiOycvOuLiCwg7Iuk6rOE7JW97J20IOyggeydgCDshLjqt7jrqLztirjsnbzsiJjroZ0g7IKs7KCE67aE7Y+s7J2Y"
           "IOu5hOykkeydtCDtgazqsowg64Ko7JWEIE1BUCDqsIAg7KCE7LK0IOqzhOyVveydmCDsgqzqs6Ag6rK97ZeY66WgCuyqveycvOuhnC"
           "DsnbTrj5ntlanri4jri6QuIOq3uOuemOyEnCDrqoXri6jsnYQgTUxFIOuhnCDqs6DrpbTrqbQg6rSA7Lih7J20IOu2gOyhse2VnCDs"
           "hLjqt7jrqLztirjqsIAsIE1BUCDroZwg6rOg66W066m0IOq0gOy4oeydtCDriITsoIHrkJwg7IS46re466i87Yq46rCAIOyEoO2Dne"
           "uQqeuLiOuLpC4K7KeA7JetIDLqsJzrpbwg67mE6rWQ7ZWgIOuVjOuKlCDqsJnsnYAg7IKs7KCE67aE7Y+s66W8IOyEuOq3uOuovO2K"
           "uCDrjIDsi6Ag7KeA7Jet7JeQIOyggeyaqe2VtCDsgqztm4TrtoTtj6wgMuqwnOulvCDrp4zrk6Tqs6AsIOq3uCAy6rCc7JeQ7IScIO"
           "y2lOy2nO2VnCDtkZzrs7jrgbzrpqwK67mE6rWQ7ZW0ICLslrTripAg7Kq97J20IOuNlCDrgpjsgaAg7ZmV66WgIiDtlZjrgpjroZwg"
           "64u17ZWp64uI64ukLgo="}
#  힌트는 (한 문장 힌트, 튜터에게 물을 것) 짝입니다 — 채점이 ❌ 인 항목에만 나옵니다
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
           "eyJtMSI6IHsi7Jew6rCEIOyCrOqzoOycqCDOu8yCICjqsbQv64WEKSI6IFsi64KY64iX7IWI7J2YIOu2hOuqqOulvCDrqLzsoIAg"
           "67O07IS47JqUIOKAlCDqs4Tslb0g7IiY66GcIOuCmOuIhOuptCDri6jsnITqsIAg6rG0L+qzhOyVveydtCDrkJjqs6Ag6rCS64+E"
           "IOygiOuwmCDsiJjspIDsnLzroZwg6rCQ7IaM7ZWp64uI64ukLiDsgqzqs6DsnKjsnZgg67aE66qo64qUIOuFuOy2nOydmCDtlans"
           "noXri4jri6QiLCAi7J20IOy9lOuTnOqwgCDrgpjriJfshYjsnZgg67aE66qo66GcIOyTtCDqsoPsnbQg66y07JeH7J246rCAIl0s"
           "ICLstJ0g64W47LacICjrhYQpIjogWyLOu8yCIOydhCDrp4zrk6Ag64KY64iX7IWI7J2YIOu2hOuqqOulvCDri6jsnIQo64WEKeyZ"
           "gCDtlajqu5gg7ZWcIO2WiSDrjZQg7Lac66Cl7ZWY66m0IOuQqeuLiOuLpCIsICLsnbQg7L2U65Oc6rCAIOu2hOuqqOuhnCDtlans"
           "grDtlZwg6rCS7J20IOyWtOuKkCDsnbTrpoTsl5DshJwg64KY7Jik64qU6rCAIl0sICJwbG90IjogWyLqsIDroZzstpUg67KU7JyE"
           "6rCAIDAuMDV+MC4yMCDsnYQg7Y+s7ZWo7ZWY64qU7KeAIMK3IOqzoeyEoOydtCDtlZjrgpjsnbjsp4Agwrcg7LWc64yT6rCSKOuY"
           "kOuKlCDstZzshp/qsJIpIOychOy5mOyXkCDsiJjsp4HshKDsnYQg6re47JeI64qU7KeAIMK3IOuRkCDstpXsl5Ag652867Ko7J2E"
           "IOuLrOyVmOuKlOyngCDssKjroYDroZwg67O07IS47JqUIiwgIuuCtCDqt7jrprzsnZgg7Ja064qQIOy2lcK37Ja064qQIOyEoOyd"
           "tCDquLDspIDqs7wg7Ja06riL64Ks64qU7KeAIl0sICJlcnJvciI6IFsi7Jik66WYIOusuOyepSDssqsg7ZaJ7J2EIOq3uOuMgOuh"
           "nCDtipzthLDsl5Ag67O07JesIOyjvOqzoCwg7KSA67mEIOyFgOydtCDrp4zrk6Ag7J2066aEKHkgwrcgRSDCtyBwb2lzc29uX25s"
           "bCnsnYAg64uk7IucIOunjOuTpOyngCDrp5Dqs6Ag6re464yA66GcIOyCrOyaqe2VmOudvOqzoCDrqoXsi5ztlZjshLjsmpQiLCAi"
           "7J20IOyYpOulmOqwgCDslrTripAg7L2U65OcIO2WieyXkOyEnCDsmZwg64Ks64qU7KeAIl19LCAibTIiOiB7IuyEuOq3uOuovO2K"
           "uCDsiJggKOqwnCkiOiBbIuOAjOyXrOq4sOu2gO2EsCBBSSDsvZTrk5zjgI0g7JWE656YIOy9lOuTnOyXkOyEnCDtkZzrpbwg66eM"
           "65Oc64qUIOynkeqzhCDtlonqs7wg6re4IO2RnOydmCDquLjsnbTrpbwg7Lac66Cl7ZWY64qUIO2WieunjCDri6Tsi5wg7J297Ja0"
           "IOuztOyEuOyalCIsICLshLjqt7jrqLztirgg7IiY64qUIOynkeqzhOqwgCDrp4zrk6Ag7ZaJ7J2YIOqwnOyImOyeheuLiOuLpCDi"
           "gJQg7ZGc7J2YIOq4uOydtOulvCDri6jsnIQg44CM6rCc44CN7JmAIO2VqOq7mCAx7ZaJIOy2nOugpe2VmOuPhOuhnSDtlITroazt"
           "lITtirjsnZgg7Lac66ClIO2YleyLneyXkCDsoIHsnLzshLjsmpQiLCAi7J20IO2RnOydmCDtlZwg7ZaJ7J20IOustOyXhyDtlZjr"
           "gpjrpbwg7IS464qUIOuLqOychOyduOqwgCJdLCAi6rCA7J6lIOuGkuydgCDshLjqt7jrqLztirjsnZgg7IKs6rOg7JyoICjqsbQv"
           "64WEKSI6IFsi44CM7Jes6riw67aA7YSwIEFJIOy9lOuTnOOAjSDslYTrnpgg7L2U65Oc7JeQ7IScIOyEuOq3uOuovO2KuOuzhCDs"
           "gqzqs6DsnKjsnYQg66eM65Oc64qUIOy9lOuTnCDtlZwg7ZaJ66eMIOuLpOyLnCDsnb3slrQg67O07IS47JqUIOKAlCDqt7gg7ZaJ"
           "7J2YIOuCmOuIl+yFiOydtCDri7XsnYQg7KCV7ZWp64uI64ukIiwgIuq3uCDtlonsnZgg67aE66qo6rCAIOq3uCDshLjqt7jrqLzt"
           "irjsnZgg6rOE7JW9IOyImOyeheuLiOuLpC4g6rG0L+uFhOydtCDrgpjsmKTroKTrqbQg67aE66qo6rCAIOuFuOy2nOydmCDtlans"
           "nbTslrTslbwg7ZWp64uI64ukIOKAlCDqs6DsuZwg7ZGc64qUIOyEuOq3uOuovO2KuOuzhCDqsJLsnYQg64W47Lac66GcIOqwgOyk"
           "ke2VtCDri6Tsi5wg7ZWp7LOk7J2EIOuVjCDsoITssrQg6rOE7JW9IOyCrOqzoOycqOqzvCDsnbzsuZjtlanri4jri6QiLCAi6re4"
           "IOy9lOuTnOqwgCDshLjqt7jrqLztirjrs4Qg7IKs6rOg7Jyo7J2YIOu2hOuqqOuhnCDsk7Qg6rKD7J20IOustOyXh+yduOqwgCJd"
           "LCAicGxvdCI6IFsi7KCQIO2VmOuCmOqwgCDshLjqt7jrqLztirgg7ZWY64KY7J24IOyCsOygkOuPhOyduOyngCDCtyDqsIDroZzs"
           "tpXsnbQg6re4IOyEuOq3uOuovO2KuOydmCDstJ0g64W47Lac7J206rOgIOuhnOq3uCDriIjquIjsnbjsp4Agwrcg7IS466Gc7LaV"
           "7J20IOyEuOq3uOuovO2KuOuzhCDsgqzqs6DsnKjsnbjsp4Agwrcg65GQIOy2leyXkCDrnbzrsqjsnYQg64us7JWY64qU7KeAIOyw"
           "qOuhgOuhnCDrs7TshLjsmpQuIOq3uOumvOydgCDtkZzrpbwg6rOg7LmY64qUIOqyg+qzvCDqsJnsnYAg7ZSE66Gs7ZSE7Yq47JeQ"
           "IO2VqOq7mCDsi5ztgrXri4jri6QiLCAi64K0IOq3uOumvOydmCDslrTripAg7LaVwrfslrTripAg64iI6riI7J20IOq4sOykgOqz"
           "vCDslrTquIvrgqzripTsp4AiXSwgImVycm9yIjogWyLsmKTrpZgg66y47J6lIOyyqyDtlonsnYQg6re464yA66GcIO2KnO2EsOyX"
           "kCDrs7Tsl6wg7KO86rOgLCDspIDruYQg7IWA7J20IOunjOuToCDsnbTrpoQoZGYgwrcgZ3JvdXBfa2V5cynsnYAg64uk7IucIOun"
           "jOuTpOyngCDrp5Dqs6Ag6re464yA66GcIOyCrOyaqe2VmOudvOqzoCDrqoXsi5ztlZjshLjsmpQiLCAi7J20IOyYpOulmOqwgCDs"
           "lrTripAg7L2U65OcIO2WieyXkOyEnCDsmZwg64Ks64qU7KeAIl19LCAibTMiOiB7IlIyMSDDlyBCMTEg7IS46re466i87Yq47J2Y"
           "IE1BUCAo6rG0L+uFhCkiOiBbIuOAjOunjOuTpCDqsoPjgI3snZggTUFQIOqzteyLneydhCDtlITroaztlITtirjsl5Ag66y47J6l"
           "7Jy866GcIOyYruqyvOuKlOyngCDtmZXsnbjtlZjshLjsmpQg4oCUIOu2hOyekOyXkCDsgqzsoITrtoTtj6zsnZgg6rCA7IOBIOyC"
           "rOqzoOyZgCDqt7gg7IS46re466i87Yq47J2YIOyLpOygnCDsgqzqs6Drpbwg642U7ZWY6rOgIDEg7J2EIOu5vOuKlOyngCwg67aE"
           "66qo7JeQIOqwgOyDgSDrhbjstpwgYmV0YV8wIOqzvCDqt7gg7IS46re466i87Yq47J2YIOyLpOygnCDrhbjstpzsnYQg642U7ZWY"
           "64qU7KeALCDqt7jrpqzqs6Agd2F0Y2gg7ZWcIO2WieunjCDsk7Drnbzqs6Ag7KCB7JeI64qU7KeAIOywqOuhgOuhnCDtmZXsnbjt"
           "lanri4jri6QiLCAi64K0IOyLneydmCDrtoTsnpDsmYAg67aE66qo7JeQIOyCrOyghOu2hO2PrOydmCDqsIDsg4Eg7IKs6rOgwrfq"
           "sIDsg4Eg64W47Lac7J20IO2VnCDrsojslKnrp4wg65Ok7Ja06rCU64qU6rCAIl0sICIyNDLqsJwg7IS46re466i87Yq4IE1BUCDs"
           "nZgg7LWc7Iaf6rCSICjqsbQv64WEKSI6IFsi6rCZ7J2AIOqzteyLneydhCBzZWdfZmluZSAyNDLtlokg7KCE67aA7JeQIOyggeya"
           "qe2VtCDsg4gg7Je07J2EIOunjOuToCDrkqQg6re4IOyXtOydmCDstZzshp/qsJLsnYQg44CM7J2066aEID0g6rCSIOuLqOychOOA"
           "jSAx7ZaJ7Jy866GcIOy2nOugpe2VmOudvOqzoCDsoIHsl4jripTsp4Ag7ZmV7J247ZWY7IS47JqUIOKAlCDtkZzrpbwg7Ya17Ke4"
           "66GcIOy2nOugpe2VmOuptCDssYTsoJDsnbQg7Ja064qQIOqwkuydhCDsnb3slrTslbwg7ZWg7KeAIOqzoOultOyngCDrqrvtlanr"
           "i4jri6QiLCAi66y07IKs6rOgIOyEuOq3uOuovO2KuOydmCDrtoTsnpDsl5Ag66y07JeH7J20IOuCqOyVhCDsnojslrTslbwg7ZWY"
           "64qU6rCAIl0sICJwbG90IjogWyLsoJAg7ZWY64KY6rCAIOyEuOq3uOuovO2KuCDtlZjrgpjsnbgg7IKw7KCQ64+E7J247KeAIMK3"
           "IOqwgOuhnOy2leydtCDshLjqt7jrqLztirjrs4QgTUxFIOydtOqzoCDshLjroZzstpXsnbQg6rCZ7J2AIOyEuOq3uOuovO2KuOyd"
           "mCBNQVAg7J247KeAIMK3IOuRkCDqsJLsnbQg6rCZ7J2AIOyngOygkOydhCDsnofripQg64yA6rCB7ISg7J2EIDAg67aA7YSwIE1M"
           "RSDstZzrjJPqsJLquYzsp4Ag6re47JeI64qU7KeAIMK3IOuRkCDstpXsl5Ag44CM66eM65OkIOqyg+OAjeyXkCDsoIHtnowg7J20"
           "66aE7J2EIOuLrOyVmOuKlOyngCDssKjroYDroZwg67O07IS47JqUIiwgIuuCtCDqt7jrprzsnZgg7Ja064qQIOy2lcK37Ja064qQ"
           "IOyEoOydtCDquLDspIDqs7wg7Ja06riL64Ks64qU7KeAIl0sICJlcnJvciI6IFsi7Jik66WYIOusuOyepSDssqsg7ZaJ7J2EIOq3"
           "uOuMgOuhnCDtipzthLDsl5Ag67O07JesIOyjvOqzoCwg7KSA67mEIOyFgOydtCDrp4zrk6Ag7J2066aEKHNlZ19maW5lIMK3IHdh"
           "dGNoIMK3IGxhbV9hbGwgwrcgYWxwaGFfMCDCtyBiZXRhXzAp7J2AIOuLpOyLnCDrp4zrk6Tsp4Ag66eQ6rOgIOq3uOuMgOuhnCDs"
           "gqzsmqntlZjrnbzqs6Ag66qF7Iuc7ZWY7IS47JqUIiwgIuydtCDsmKTrpZjqsIAg7Ja064qQIOy9lOuTnCDtlonsl5DshJwg7Jmc"
           "IOuCrOuKlOyngCJdfSwgIuqzvOygnCI6IHsiTUxFIOuhnCDqs6Drpbgg7IOB7JyEIOyXtCDshLjqt7jrqLztirjsnZgg6rOE7JW9"
           "IOyImCAo6rG0KSI6IFsi66y07IKs6rOg7JyoKDEgLSDsgqzqs6Ag6rK97ZeY66WgKeydtCDrhpLsnYAg7Iic7Jy866GcIOyEuOq3"
           "uOuovO2KuCAxMOqwnOulvCDqs6Drpbgg65KkIOq3uCAxMOqwnOydmCDqs4Tslb0g7IiY66W8IOuNlO2VtCDri6jsnIQg44CM6rG0"
           "44CNIOqzvCDtlajqu5ggMe2WiSDstpzroKXtlZjrqbQg65Cp64uI64ukIOKAlCDsgqzqs6Ag6rK97ZeY66Wg7J20IOqwgOyepSDr"
           "gq7snYAg7IS46re466i87Yq4IDEw6rCc66W8IOqzoOultOuKlCDqsoPqs7wg6rCZ7Iq164uI64ukIiwgIuustOyCrOqzoOycqCDs"
           "g4HsnIQg7IS46re466i87Yq4IDEw6rCc66W8IOustOyXh+ycvOuhnCDqs6jrnpDqs6Ag66y07JeH7J2EIOuNlO2WiOuKlOqwgCJd"
           "LCAiTUFQIOuhnCDqs6Drpbgg7IOB7JyEIOyXtCDshLjqt7jrqLztirjsnZgg6rOE7JW9IOyImCAo6rG0KSI6IFsi6rCZ7J2AIOyd"
           "vOydhCBNQVAg7Je066GcIO2VnCDrsogg642UIO2VqeuLiOuLpCDigJQg66qF64uo7J20IO2GteynuOuhnCDrsJTrgIzripQg6rKD"
           "7J2EIO2ZleyduO2VmOuKlCDqsoPsnbQg7J20IOqwkuydmCDrqqnsoIHsnbTrnbwsIE1MRSDroZwg6rOg66W4IOqwkuqzvCDrgpjr"
           "noDtnogg7Lac66Cl7ZW0IOuRkOyEuOyalCIsICLrkZAg66qF64uo7J20IOyZnCDri6Trpbgg7IS46re466i87Yq466W8IOqzoOul"
           "tOuKlOqwgCJdLCAi66y07IKs6rOgIO2ZmOq4iSDrjIDsg4Eg6rOE7JW9IOyImCAo6rG0KSI6IFsi7KCE7LK0IOqzhOyVveydmCDs"
           "gqzqs6Ag6rK97ZeY66Wg7J2EIDEg7JeQ7IScIOu6gCDrrLTsgqzqs6Ag67mE7Jyo7JeQIOyghOyytCDqs4Tslb0g7IiY66W8IOqz"
           "se2VmOqzoCDjgIzrjIDsg4Eg6rOE7JW9IOyImCA9IDAwMCwwMDAg6rG044CNIO2YleyLneycvOuhnCAx7ZaJIOy2nOugpe2VmOup"
           "tCDrkKnri4jri6Qg4oCUIOu5hOycqOyXkCDruYTsnKjsnYQg6rOx7ZWY7KeAIOyViuyVmOuKlOyngCDri6jsnITroZwg6rKA7IKw"
           "7ZWY7IS47JqUIiwgIuydtCDsiKvsnpDsnZgg64uo7JyE6rCAIOu5hOycqOyduOqwgCDqsbTsnbjqsIAiXSwgIlI0MiDqsIAgUjQz"
           "IOuztOuLpCDrhpLsnYQg7ZmV66WgICjtmZXrpaApIjogWyLrkZAg7KeA7JetIO2RnCByZWdfYmluIOyXkOyEnCBSNDMg6rO8IFI0"
           "MiDrkZAg7ZaJ66eMIOqzqOudvCwg6re4IO2WieydmCDsgqzqs6Ag6rOE7JW9IOyImCBzIOyZgCDqs4Tslb0g7IiYIG4g7Jy866Gc"
           "IOyCrO2bhOu2hO2PrOulvCDtlZjrgpjslKkg66eM65Oc7IS47JqUIiwgIuyCrO2bhOu2hO2PrOuKlCBCZXRhKM6xX2IgKyBzLCDO"
           "sl9iICsgbiDiiJIgcykg7J6F64uI64ukIOKAlCDrkZAg67aE7Y+s7JeQ7IScIOqwmeydgCDsiJjrp4ztgbwg7LaU7Lac7ZW0IFI0"
           "MiDsqr3snbQg7YGwIO2RnOuzuOydmCDruYTsnKjsnYQg6rOE7IKw7ZWY66m0IOuQqeuLiOuLpC4g6rCS7J20IDAuMDIg6re87LKY"
           "66m0IOuRkCDsp4Dsl63snYQg67CY64yA66GcIOyngOygle2VnCDqsoPsnbTqs6AsIDAuNzEg6re87LKY66m0IOy4oeyglSDsp4Dt"
           "kZzrpbwg7IKs6rOgIOqxtOyImCDDtyDrhbjstpzroZwg67CU6r+UIOqzhOyCsO2VnCDqsoPsnoXri4jri6QiLCAi65GQIOyCrO2b"
           "hOu2hO2PrOulvCDrrLTsl4fsnLzroZwg66eM65Ok7JeI6rOgIO2ZleuloOydhCDslrTrlrvqsowg6rOE7IKw7ZaI64qU7KeAIl0s"
           "ICJwbG90IjogWyLsoJAg7ZWY64KY6rCAIOyEuOq3uOuovO2KuCDtlZjrgpjsnbgg7IKw7KCQ64+E7J247KeAIMK3IOqwgOuhnOy2"
           "leydtCDshLjqt7jrqLztirjrs4QgTUxFIOydtOqzoCDshLjroZzstpXsnbQg7IS46re466i87Yq467OEIE1BUCDsnbjsp4Agwrcg"
           "65GQIOqwkuydtCDqsJnsnYAg7KeA7KCQ7J2EIOyeh+uKlCDrjIDqsIHshKDsnYQg7KCQ7J20IOu2hO2PrO2VnCDqtazqsIQg7KCE"
           "7LK07JeQIOq3uOyXiOuKlOyngCDCtyDrkZAg7LaV7JeQIOudvOuyqOydhCDri6zslZjripTsp4Ag7LCo66GA66GcIOuztOyEuOya"
           "lCIsICLrgrQg6re466a87J2YIOyWtOuKkCDstpXCt+yWtOuKkCDshKDsnbQg6riw7KSA6rO8IOyWtOq4i+uCrOuKlOyngCJdLCAi"
           "ZXJyb3IiOiBbIuyYpOulmCDrrLjsnqUg7LKrIO2WieydhCDqt7jrjIDroZwg7Yqc7YSw7JeQIOuztOyXrCDso7zqs6AsIOykgOu5"
           "hCDshYDsnbQg66eM65OgIOydtOumhChzZWdfYmluIMK3IHBfZ2xvYmFsIMK3IG51IMK3IHJlZ19iaW4gwrcgcm5nKeydgCDri6Ts"
           "i5wg66eM65Ok7KeAIOunkOqzoCDqt7jrjIDroZwg7IKs7Jqp7ZWY65286rOgIOuqheyLnO2VmOyEuOyalCIsICLsnbQg7Jik66WY"
           "6rCAIOyWtOuKkCDsvZTrk5wg7ZaJ7JeQ7IScIOyZnCDrgqzripTsp4AiXX19")
_ASK = {"m1": "full", "m2": "blank",       # 뒤로 갈수록 걷힙니다 — 미션 3 은 틀만 남습니다
        "m3": "self", "과제": "bare"}
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 명시합니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 틀 (문장으로 만드는 것은 내가 씁니다)",
             "self": "◼ 스스로 확인할 항목의 틀 (이 미션은 힌트 없이 내가 명세합니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " 「여기부터 AI 코드」 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. 「여기부터 AI 코드」 아래 코드에서"
                      " 「{ask}」 부터 확인하고, 어느 코드 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : 「여기부터 AI 코드」 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 2단 힌트가 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 행만 고릅니다 — 단위가 붙은 행, 또는 「이름 = 값」 꼴의 행.

    표를 통째로 띄우면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 단위와 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 단위를 달아 고른 행이 하나라도 있으면 그
    행들만 보고, 그런 행이 아예 없을 때만 출력 전체를 봅니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 길을 열어 두지 않으면
    `plt.vlines(...)` 로 화면에 분명히 그어 놓은 수직선이 「수직선이 아예 없다」 로 채점돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·채움 영역(PolyCollection)은 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _lines(ax):
    """축의 선을 곡선(점 10개 이상)과 수직선(x 가 같은 두 점)으로 갈라 돌려줍니다"""
    cur, vline = [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 깨져도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선(add_changepoints_to_plot)은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.vlines` 로 그은 수직선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size >= 10:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 놓인 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)                     # LineCollection 으로 그은 y = x 기준선도 셉니다
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 놓인 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


def _near(val, ref, factor=1.5):
    """실측값이 기준 기준의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, vline = _lines(ax)
        want = spec.get("n_series", len(cur))
        if len(cur) != want:
            gaps.append(f"곡선이 {len(cur)}개다 (기준 {want}개)")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.3g}~{xd.max():.3g} 만 덮는다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])         # 로그우도로 그렸으면 최댓값이 그 자리다
                trough = float(xd[yd.argmin()])         # NLL 로 그렸으면 최솟값이 그 자리다
                if not (p0 <= peak <= p1 or p0 <= trough <= p1):
                    gaps.append(f"곡선의 최댓값이 {peak:.4f}, 최솟값이 {trough:.4f} 자리다"
                                f" (기준 {p0}~{p1} — 둘 중 하나가 그 구간에 들어야 합니다)")
        if cur and ("봉우리_x" in spec or "골짜기_x" in spec):
            # 부호가 뜻을 가진 곡선(충격반응) — 위로 솟는 곡선과 아래로 꺼지는 곡선을 따로 봅니다.
            # 반응을 통째로 뒤집어 그리면 최댓값·최솟값의 자리는 그대로라 자리만 봐서는 걸러지지 않습니다
            if "봉우리_x" in spec:
                a0, a1 = spec["봉우리_x"]
                if not any(float(np.nanmax(y)) > 0 and a0 <= float(x[int(np.nanargmax(y))]) <= a1
                           for x, y in cur):
                    gaps.append(f"위로 솟아 {a0:g}~{a1:g} 시간 자리에서 가장 높아지는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
            if "골짜기_x" in spec:
                b0, b1 = spec["골짜기_x"]
                if not any(float(np.nanmin(y)) < 0 and b0 <= float(x[int(np.nanargmin(y))]) <= b1
                           for x, y in cur):
                    gaps.append(f"아래로 꺼져 {b0:g}~{b1:g} 시간 자리에서 가장 낮아지는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
        if "vline_x" in spec:
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append("최댓값 자리를 짚는 수직선이 기준 자리에 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (수직선이 아예 없다)"))
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec and len(heights) != spec["n"]:
            gaps.append(f"막대가 {len(heights)}개다 (기준 {spec['n']}개)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다 (기준 {spec['ymax']})")
    elif spec.get("kind") == "scatter":
        markers = _marker_lines(ax)                              # plot(x, y, "o") 로 찍은 점도 셉니다
        pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
              for c in ax.collections if not hasattr(c, "get_segments")]   # 선 컬렉션(LineCollection)은 점으로 세지 않습니다
        pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in markers]
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            n_spec = spec["n_points"]
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 세그먼트 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
            if "gap_min" in spec:
                spread = float(np.abs(ys - xs).max())
                if spread < spec["gap_min"]:
                    gaps.append(f"두 축의 값이 가장 크게 벌어진 곳도 {spread:.4f} 뿐이다"
                                f" (기준 {spec['gap_min']:g} 이상)")
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in markers], xy=xy):
            gaps.append("두 축의 값이 같은 자리를 잇는 대각선(y = x)이 점이 놓인 구간에 없다")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


def _ref_plot_m1():
    """미션 1 의 참조 그림 — 「준비 셀」의 y · E · poisson_nll 로 로그우도 곡선을 그립니다"""
    y_, E_, nll = _get("y"), _get("E"), _get("poisson_nll")
    if y_ is None or E_ is None or nll is None:
        return None
    grid = np.linspace(0.05, 0.20, 160)
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    a.plot(grid, np.array([-nll(g) for g in grid]), color=COL["파랑"])
    a.axvline(float(np.sum(y_) / np.sum(E_)), color=COL["하늘색"], ls="--")
    a.set_xlabel("연간 사고율 $\\lambda$ (건/년)")
    a.set_ylabel("로그우도 $\\ell(\\lambda)$")
    return f


def _ref_plot_m2():
    """미션 2 의 참조 그림 — 「준비 셀」의 df · group_keys 로 242개 세그먼트의 깔때기 산점도를 그립니다"""
    d, keys = _get("df"), _get("group_keys")
    if d is None or not keys:
        return None
    tbl = (d.groupby(list(keys), observed=True)
           .agg(claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
           .reset_index())
    tbl["lam"] = tbl["claims"] / tbl["exposure"]
    is_small_sample = tbl["exposure"] < 100                    # 관측이 적은 세그먼트를 하늘색으로 구분해 둡니다
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    a.scatter(tbl.loc[~is_small_sample, "exposure"], tbl.loc[~is_small_sample, "lam"], s=12, color=COL["연회색"])
    a.scatter(tbl.loc[is_small_sample, "exposure"], tbl.loc[is_small_sample, "lam"], s=12, color=COL["하늘색"])
    a.set_xscale("log")
    a.set_xlabel("세그먼트 총 노출 (년, 로그 눈금)")
    a.set_ylabel("연간 사고율 MLE (건/년)")
    return f


def _ref_plot_m3():
    """미션 3 의 참조 그림 — 「준비 셀」의 seg_fine · alpha_0 · beta_0 로 MLE 대 MAP 산점도를 그립니다"""
    tbl, a0, b0 = _get("seg_fine"), _get("alpha_0"), _get("beta_0")
    if tbl is None or a0 is None or b0 is None:
        return None
    mle = tbl["claims"] / tbl["exposure"]
    mapv = (a0 + tbl["claims"] - 1) / (b0 + tbl["exposure"])
    is_small_sample = tbl["exposure"] < 100                    # 관측이 적은 세그먼트를 하늘색으로 구분해 둡니다
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    a.scatter(mle[~is_small_sample], mapv[~is_small_sample], s=12, color=COL["연회색"])
    a.scatter(mle[is_small_sample], mapv[is_small_sample], s=12, color=COL["하늘색"])
    axis_end = float(mle.max())
    a.plot([0, axis_end], [0, axis_end], lw=1, color=COL["회색"])    # 두 추정치가 같은 자리 — 대각선
    a.set_xlabel("MLE — 데이터가 말한 그대로 (건/년)")
    a.set_ylabel("MAP — 감마 사전분포를 반영한 뒤 (건/년)")
    return f


def _ref_plot_task():
    """과제의 참조 그림 — 「준비 셀」의 seg_bin · p_global · nu 로 MLE 대 MAP 산점도를 그립니다"""
    tbl, p0, v = _get("seg_bin"), _get("p_global"), _get("nu")
    if tbl is None or p0 is None or v is None:
        return None
    a_b, b_b = 1 + p0 * v, 1 + (1 - p0) * v
    mle = tbl["s"] / tbl["n"]
    mapv = (a_b + tbl["s"] - 1) / (a_b + b_b + tbl["n"] - 2)
    is_small_sample = tbl["n"] < 100                            # 계약이 적은 세그먼트를 하늘색으로 구분해 둡니다
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    a.scatter(mle[~is_small_sample], mapv[~is_small_sample], s=12, color=COL["연회색"])
    a.scatter(mle[is_small_sample], mapv[is_small_sample], s=12, color=COL["하늘색"])
    axis_end = float(mle.max())
    a.plot([0, axis_end], [0, axis_end], lw=1, color=COL["회색"])    # 두 추정치가 같은 자리 — 대각선
    a.set_xlabel("MLE — 데이터가 말한 그대로 (비율)")
    a.set_ylabel("MAP — 베타 사전분포를 반영한 뒤 (비율)")
    return f


_REF = {"m1": _ref_plot_m1, "m2": _ref_plot_m2,   # 미션마다 참조 그림을 그리는 함수 하나
        "m3": _ref_plot_m3, "과제": _ref_plot_task}


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 결과는 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 문장으로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt



def _prompt1(n=70):
    """지금 셀의 지시 첫 행 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _prompt_missing():
    """지시 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    try:
        txt = str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "").strip()
    except Exception:
        txt = str(_get("PROMPT") or "").strip()
    if not txt:
        return "PROMPT 입력란이 비어 있습니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    stripped = txt.replace("—", "").replace("·", "").strip()
    if stripped in _PROMPT_PLACEHOLDERS:
        return "PROMPT 입력란이 자리표시자 그대로입니다"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 행부터 채점 표를 닫는 `─` 구분선 행까지를
    출력 패널이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing()                        # 값·그래프보다 먼저 — 내 말로 시킨 자리가 있는가
    _locked = bool(reason)
    if reason and key != "m2":
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션만은 전임 담당자 코드의 진단이 먼저다
        print(f"ⓘ {reason} — 아래 채점 결과는 전임 담당자 코드의 진단입니다."
              " 고칠 지시를 PROMPT 에 쓰세요")
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _HINT.get(key, {}).get("error", ("", "이 오류가 어느 코드 행에서 왜 났는지"))
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 행이 있으면 그 행만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) > 30        # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 띄운 모양
    if table_flood:
        print(f"❌ 값 — 단위도 「이름 = 값」 도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를"
              " 통째로 띄우면 어느 숫자가 답인지 채점이 고르지 못합니다."
              " 값은 「이름 = 숫자 단위」 한 행으로 출력하세요")
        nums = []
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        tail = _label_lines(val_lines, label) if nums else []   # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
        fell_back = bool(nums) and not tail             # 라벨이 든 행이 없을 때만 값줄 전체로 물러납니다
        hits = [v for v in (_nums("\n".join(tail)) if tail else nums) if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif table_flood:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 「이름 = 숫자 {unit}」 한 행으로 골라 출력하면 채점이 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준에 닿는 숫자를 출력에서 찾지 못했습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            gaps = None
            for ax in axs:                 # 패널이 여럿이면 기준에 맞는 축이 하나만 있어도 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    gaps = []
                    break
                if gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if gaps:
                passed = False
                failed.append(("그래프", "plot"))
                print("❌ 그래프 — " + " · ".join(gaps))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    tbl = _HINT.get(key, {})
    attempt = _TRY[key] = _TRY.get(key, 0) + (1 if failed else 0)
    for name, hkey in failed:
        tiers = tbl.get(hkey, ("",))
        tier = tiers[:-1] if len(tiers) > 2 else tiers[:1]   # 2단 힌트는 마지막 원소가 튜터에 물을 것입니다
        hint = tier[min(attempt, len(tier)) - 1] if tier else ""
        if hint:
            print(f"   힌트({name}) : {hint}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[-1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(), ask=question))
    elif _locked:                                  # 값·그림은 닿았지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 기준에 닿았습니다 — PROMPT 에 고칠 지시를 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 행이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                           # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」이면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 출력됩니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 로그우도 최댓값의 위치를 소수점까지 확정할 수 있을까

### 미션 1 — 포트폴리오 사고율 λ̂ 의 점추정

#### 할 일 — 미션 1 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
3. 셀을 클릭한 채 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶.
4. 실행 결과(출력)에 나오는 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문** — 계약 678,013건을 하나의 집단으로 두면 연간 사고율은 얼마입니까?

이론 4절에서 전체 계약의 연간 사고율 0.1006건/년을 공식으로 이미 계산했습니다. 미션 1 은 같은 값을 음의 로그우도와 수치 최적화로 다시 계산해 소수점 아래 6자리까지 대조합니다.

#### 실행 전에 — 계획을 골라 보세요

포트폴리오의 연간 사고율을 구하는 절차로 어느 쪽을 쓸까요? 하나를 고르고, 왜 그것이 보고서의 근거가 되는지 한 문장 적으세요.

> **[예측] 포트폴리오의 연간 사고율을 구하는 절차로 어느 쪽을 쓸까요?**
>
> 1. 계약당 사고 건수의 평균을 그대로 보고한다
> 2. 사고 건수의 합을 노출의 합으로 나눈다
> 3. 포아송 로그우도를 적어 그 최댓값을 주는 $\lambda$를 수치 최적화로 찾는다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

추가로, 3개 중 **2개는 같은 값이 계산됩니다**. 어느 2개인지도 함께 적어 두세요.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `y` | 계약별 사고 건수, 678,013건 · 단위 건 | 고정 |
| `E` | 계약별 노출, 단위 년 | 고정 |
| `poisson_nll(lam)` | 사고율 후보 하나를 받아 음의 로그우도를 반환하는 함수 | 그대로 부른다 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다.

1. **연간 사고율 λ̂** — 단위 건/년 · 소수점 아래 6자리 · `poisson_nll` 을 수치 최적화로 가장 작게 만드는 사고율
   - **음의 로그우도(NLL)** 는 로그우도에 마이너스를 곱해 최댓값 찾기를 최솟값 찾기로 바꾼 값입니다. 부호만 바꾸었으므로 최적해의 가로축 위치, 곧 $\lambda$ 는 변하지 않습니다
2. **총 노출** — 단위 년 · 소수점 아래 1자리 · `E` 를 전부 더한 값이고, 1번 나눗셈의 분모에 해당합니다
3. **추가할 한 행** — 최적화의 수렴 여부와 반복 횟수
   - 채점 대상은 아니고, 통과 뒤 열리는 완성본과 대조할 항목입니다
4. **그래프 한 장** — 가로축은 후보 연간 사고율 0.05 ～ 0.20, 세로축은 로그우도인 선 그래프
   - 곡선 : 후보를 촘촘히 나열한 로그우도 곡선 하나 — `poisson_nll` 에 마이너스를 곱한 값입니다
   - 수직선 : 1번에서 찾은 사고율 위치에 세로축 아래 끝부터 위 끝까지 곧은 선 하나 — 곡선은 로그우도 선 한 개뿐이고, 이 수직선은 곡선으로 세지 않습니다
   - 가로축 이름 : 「연간 사고율 $\lambda$ (건/년)」
   - 세로축 이름 : 「로그우도 $\ell(\lambda)$」

**출력 형식** — 「이름 = 값 단위」 형식으로 1행씩 `print` 합니다. 그래프는 한 장입니다.

통과하면 공식으로 바로 구하는 해(closed-form)와 수치 최적화 2가지가 같은 값을 주는지 소수점 아래 6자리까지 대조해 보세요. 이 대조가 미션 1 의 확인 항목입니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `y` · `E` · `poisson_nll` 은 이미 있으니 다시 만들지 말라고 적습니다.
- **최소화 대상** — 어느 함수를 가장 작게 만드는 사고율을 찾는지, 그리고 후보가 0 보다 크다는 것.
- **출력** — 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 가로축 범위 · 세로축이 로그우도라는 것 · 수직선을 그을 위치 · 가로축·세로축 이름.

**막히면** — AI 튜터에게 「최소화 함수에 목적함수와 시작값을 어떻게 전달하나요」 처럼 물어보세요.

<details class="lms-extra"><summary>음의 로그우도를 최소화하는 이유</summary>

최적화 도구는 대개 최솟값을 찾도록 만들어져 있습니다. 그래서 최댓값을 찾는 문제를 부호만 바꿔 최솟값을 찾는 문제로 전달합니다 — 어제 4절에서 적은 로그우도를 그대로 쓰되, `minimize` 에 전달할 목적함수만 음의 로그우도로 둡니다. 세로축을 음의 로그우도로 그리면 곡선이 상하로 반전되어, 로그우도의 최댓값 지점이 그대로 최솟값 지점이 됩니다. 가로축 위치는 변하지 않으므로 수치 최적화가 찾은 $\lambda$ 도 같습니다.

부호만 바뀌므로 높이의 절댓값도 같습니다 — 완성본이 출력하는 로그우도 −146,624 와 음의 로그우도 +146,624 는 같은 지점의 두 표기입니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Nelder-Mead_Rosenbrock.gif" width="440" height="440"/>
<p align="center"><em>▲ 수치 최적화가 후보값을 반복 계산하며 최솟값으로 이동하는 과정 — 빨간 삼각형이 후보값을 갱신하며 최솟값 위치로 수렴한다. 검은 곡선은 목적함수의 등고선이고, 우리가 <code>minimize</code> 에 전달할 목적함수가 바로 음의 로그우도다</em><br><span style="font-size:0.9em;color:#656565">출처: Wikipedia, Nelder-Mead_Rosenbrock.gif</span></p>

</details>

<details class="lms-extra"><summary>주의 — AI 에 위임할 것과 직접 확인할 것</summary>

도우미가 코드를 넣는 곳은 마지막으로 클릭한 코드 셀입니다. 다른 셀을 클릭한 채 시키면 엉뚱한 셀이 바뀝니다.

왼쪽은 문법과 기본 구조 작성이라 맡겨도 되는 일이고, 오른쪽은 **틀려도 오류가 나지 않아** 사람이 봐야 하는 일입니다.

| AI 에게 맡길 일 | 내가 반드시 확인할 일 |
|---|---|
| 최적화 호출 코드와 인자 채우기 | 최소화 대상이 **음의** 로그우도인가 |
| 후보 $\lambda$ 몇 개의 값을 나란히 출력 | 나눗셈의 분모가 계약 수가 아니라 **노출의 합**인가 |
| 로그우도 곡선을 그리고 수직선 긋기 | 수렴에 성공했는가, 값의 단위가 건/년으로 타당한가 |

</details>

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 변수를 직접 만듭니다
from sklearn.datasets import fetch_openml

# freMTPL2freq — 프랑스 자동차보험 계약별 사고 빈도 (678,013 x 12). OpenML 에서 바로 내려받습니다
df = fetch_openml(data_id=41214, as_frame=True).frame
df["IDpol"] = df["IDpol"].astype(int)
for c in ["Area", "VehBrand", "VehGas", "Region"]:
    df[c] = df[c].astype(str).str.strip("'").astype("category")

# 어제 정해 5일 내내 고정하는 전처리 규칙 2개
df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)   # 사고 5건 이상은 극소수라 상한 4 로 절사
df["Exposure"] = df["Exposure"].clip(upper=1.0)           # 노출은 1년 상한

y = df["ClaimNb"].to_numpy(dtype=float)      # 쓸 변수 1 — 계약별 사고 건수 (단위 건)
E = df["Exposure"].to_numpy(dtype=float)     # 쓸 변수 2 — 계약별 노출 (단위 년)


def poisson_nll(lam):                        # 쓸 변수 3 — 이론에서 만든 노출 보정 포아송의 음의 로그우도
    """lam 은 숫자 하나 또는 길이 1 배열. 최소화용이라 로그우도에 마이너스를 붙여 돌려줍니다"""
    lam = float(np.atleast_1d(lam)[0])
    if lam <= 0:
        return 1e12                          # 범위 밖 후보에는 큰 벌점
    return -np.sum(y * np.log(lam * E) - lam * E - gammaln(y + 1))


print("행 x 열 :", df.shape, "| 결측치 총합 :", int(df.isna().sum().sum()))
print(df[["IDpol", "ClaimNb", "Exposure", "Area", "VehBrand", "Region"]].head())
print("쓸 수 있는 이름 : y (사고 건수, 건) · E (노출, 년) · poisson_nll(lam) (음의 로그우도)")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 미션 1 의 「여기부터 AI 코드」 아래 코드에서 `minimize` 에 넘긴 목적함수가 `poisson_nll` 인 것이 결과에 어떻게 작용하나요?**
>
> 1. 로그우도를 그대로 넘긴 것과 같아, 최적화가 로그우도의 최댓값을 향해 올라간다
> 2. 부호를 뒤집어 최솟값을 찾게 만든 것이라, 찾은 자리는 로그우도가 가장 큰 λ 와 같은 가로 위치다
> 3. 부호를 뒤집었으므로 찾은 λ 도 뒤집혀, 로그우도 최대 지점과는 다른 값이 나온다

## 세그먼트를 더 세분하면 요율표가 더 공정해질까

### 미션 2 — 지역 × 브랜드 요율표 (함정 미션)

#### 할 일 — 미션 2 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 「PROMPT 셀」을 **먼저 그대로 ▶** 해 전임 담당자 코드의 채점 결과(❌)를 봅니다.
3. 아래 고칠 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
4. 셀을 클릭한 채 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶.
5. 실행 결과(출력)에 나오는 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 4번부터 다시 합니다.

**이 미션의 질문** — 지역 × 브랜드 세그먼트마다 연간 사고율을 적으면 값의 범위가 얼마나 됩니까?

전체 계약의 사고율 한 값만 보고했더니 팀장이 되묻습니다 — **"모든 계약에 같은 사고율을 매기면, 사고가 드문 시골 운전자가 대도시 운전자의 보험금을 대신 내 주는 구조 아닌가요?"** 그래서 이번에는 **세그먼트**(segment)마다 사고율을 따로 계산합니다. 세그먼트는 위험 수준이 비슷한 계약의 집합이고, 여기서는 지역과 브랜드가 같은 계약이 한 세그먼트입니다. 지역 22개와 브랜드 11개의 조합 중 계약이 실제로 있는 것만 남기므로 세그먼트 수는 만들 것 2번에서 계산합니다. 쓰는 식은 **이론 4절**의 노출 보정 추정 $\hat{\lambda} = \sum_i y_i / \sum_i E_i$ 그대로이고, 합을 내는 범위만 그 세그먼트 안으로 좁힙니다.

#### 고치기 전에 — 틀린 곳을 먼저 찾아보세요

「여기부터 AI 코드」 아래 코드를 한 행씩 읽고, 값이 아니라 **단위**로 검산해 보세요.

> **[예측] 아래 AI 코드는 실행되는데 답이 틀렸습니다. 틀린 곳은 어디이고 왜일까요?**
>
> 1. 집계 코드 행 — `.agg(...)` 가 사고 건수를 합이 아니라 개수로 세고 있어서
> 2. 사고율 코드 행 — 나눗셈의 분모가 노출의 합이 아니라 계약 수여서 단위가 건/계약이 된다
> 3. 출력 코드 행 — `.max()` 가 가장 높은 세그먼트가 아니라 표 전체의 최댓값을 읽고 있어서
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

지적한 곳에 근거 1문장을 적으세요.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `df` | 계약 678,013행 표, 열은 `ClaimNb` · `Exposure` · `Region` · `VehBrand` | 고정 |
| `group_keys` | 세그먼트 기준 열 2개 `Region` · `VehBrand` | 고정 |
| `lam_all` | 전체 계약(회사 전체)의 연간 사고율, 단위 건/년 · 대조용 | 고정 |
| `claims_all` | 원본 전체 사고 건수, 단위 건 · 대조용 | 고정 |

#### 고칠 것

「여기부터 AI 코드」 행 아래에 전임 담당자 코드가 이미 들어 있습니다. 오류 없이 실행되지만 값이 틀렸고 그림이 없습니다. 고친 뒤의 아래 3개가 이 미션에서 만들 것입니다.

1. **가장 높은 세그먼트의 사고율** — 단위 건/년 · 소수점 아래 6자리
   - 지금 : 분모가 잘못 지정되어 값이 절반 아래로 감소합니다
   - 고친 뒤 : 같은 사고 합을 그 세그먼트의 노출 합 `exposure` 로 나눕니다 — 계약 수로 나누면 단위가 건/계약이라 사고율이 아닙니다
   - 그 세그먼트의 이름도 「지역 × 브랜드」 형태로 같은 행에 함께 출력합니다. 다음 미션의 배경에서 이 세그먼트와 바로 아래 세그먼트를 함께 다룹니다
2. **세그먼트 수** — 단위 개 · 정수
   - 지금 : 표의 행 수를 세어 맞게 나옵니다. 이 행은 그대로 둡니다
3. **그래프 한 장** — 가로축은 세그먼트의 총 노출, 세로축은 그 세그먼트의 연간 사고율인 산점도
   - 점 : 세그먼트 하나에 점 하나, 모두 242개
   - 가로축 척도 : 로그 척도 — 눈금 한 단계마다 값이 10배가 되는 척도이고, matplotlib 에서는 `ax.set_xscale("log")` 한 행입니다
   - 가로축 이름 : 「세그먼트 총 노출 (년, 로그 눈금)」
   - 세로축 이름 : 「연간 사고율 MLE (건/년)」
   - 지금 : 그림이 없습니다. 같은 프롬프트에서 새로 시켜야 나옵니다
   - 아래 깔때기 그림이 우리가 그릴 모양입니다

<img src="https://en.wikipedia.org/wiki/Special:FilePath/FloreAndWicherts2015_meta_analysis_sex_stereotype_threat.png" width="480" height="449"/>
<p align="center"><em>▲ 깔때기 그림(funnel plot) — 표본이 작은 연구일수록 추정값이 좌우로 넓게 흩어진다. 우리가 그릴 그림은 가로축에 총 노출, 세로축에 사고율을 배치해 같은 깔때기가 왼쪽에서 넓어진다</em><br><span style="font-size:0.9em;color:#656565">출처: Wikimedia Commons, Flore &amp; Wicherts 2015 meta-analysis funnel plot</span></p>

**통과 기준** — 값 2개가 「이름 = 값 단위」 형식으로 1행씩 출력되고, 산점도 한 장이 위 요구를 다 갖추면 통과입니다.

#### 프롬프트에 넣을 것(힌트)

- **고칠 곳** — 세그먼트별 사고율을 만드는 나눗셈 1행, 그 분모를 무엇으로 바꿀지.
- **수정하지 말 범위** — `df` 와 `group_keys` 는 이미 있으니 데이터를 다시 읽지 말고 그대로 쓰라고 적습니다.
- **출력** — 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 가로축·세로축에 무엇을 배치할지 · 가로축 척도 · 이름을 다는 것.

**막히면** — AI 튜터에게 「세그먼트별 사고율의 분모를 노출의 합으로 바꾸면 단위가 어떻게 달라지나요」 처럼 물어보세요.

<details class="lms-extra"><summary>가로축을 로그 척도로 두는 이유</summary>

세그먼트마다 총 노출이 2.4년부터 3만 년 넘게까지 분포합니다. 선형 척도에서는 점의 대부분이 왼쪽 끝에 밀집해 분포 형태가 드러나지 않습니다. 로그 척도는 눈금 한 단계마다 값이 10배가 되는 척도라, 노출이 1만 배 차이 나는 세그먼트도 한 화면에 고르게 배치됩니다.

눈금은 10 · 100 · 1,000 · 10,000 처럼 읽습니다. 가로축에서 오른쪽 눈금 한 단계는 그 세그먼트에 누적된 관측이 10배라는 뜻이고, 오른쪽으로 갈수록 세로축 방향 산포가 좁아지는 것이 이 미션에서 확인할 형태입니다. 어제 4절의 포아송 MLE $\hat{\lambda} = \bar{x}$ 에서 분모만 노출의 합으로 바꾼 식이라, 분모가 커질수록 추정값의 산포가 감소합니다.

</details>

<details class="lms-extra"><summary>깔때기 그림의 가로축·세로축과 점선</summary>

검은 점 하나가 연구 하나입니다. 가로축은 각 연구의 추정값이고, 세로축은 그 추정값의 표준오차(표본마다 추정값이 흩어지는 정도)라 표본이 클수록 0 에 가까워집니다. 그래서 위쪽에서는 점이 가운데로 모이고 아래로 갈수록 좌우로 넓게 흩어집니다. 점선과 회색 영역은 원 논문의 유의수준(귀무가설을 기각하는 기준) 표시라 지금은 넓어지는 형태만 확인하면 됩니다.

</details>

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 변수를 직접 만듭니다
if "df" not in globals():                                       # 이 미션만 따로 열어도 이 셀부터 실행하면 됩니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)       # 어제 정한 전처리 규칙 2개 — 미션 1 과 같습니다
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

group_keys = ["Region", "VehBrand"]                           # 쓸 변수 1 — 세그먼트 기준 열 2개
lam_all = float(df["ClaimNb"].sum() / df["Exposure"].sum())   # 쓸 변수 2 — 전체 계약의 연간 사고율 (건/년), 대조용
claims_all = int(df["ClaimNb"].sum())                         # 쓸 변수 3 — 원본 전체 사고 건수 (건), 대조용

print("세그먼트 축 :", group_keys, "|",
      f'지역 {df["Region"].nunique()}개 x 브랜드 {df["VehBrand"].nunique()}개'
      " — 계약이 실제로 있는 조합이 몇 개인지는 표를 만들어 보면 나옵니다")
print(f"대조용 — 전체 계약 사고율 lam_all : {lam_all:.6f} 건/년"
      f" | 원본 사고 합 claims_all : {claims_all:,} 건")
print("쓸 수 있는 이름 :  df · group_keys · lam_all · claims_all")

In [ ]:
# 미션 2 — 프롬프트 (함정)   # ai: prompt
# 이 미션은 함정입니다 — 아래는 전임 담당자가 남긴 코드라 돌아가지만 답이 틀렸습니다.
# 어디를 어떻게 고칠지 PROMPT 에 쓰면 AI 튜터가 「여기부터 AI 코드」 아래를 다시 씁니다.
PROMPT = """

"""
# --- 여기부터 AI 코드 ---
# ↓ 전임 담당자의 코드 — 그대로 실행하면 채점이 ❌ 를 냅니다
seg_fine = (df.groupby(group_keys, observed=True)
              .agg(n=("IDpol", "size"), claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
              .reset_index())
seg_fine["lam_mle"] = seg_fine["claims"] / seg_fine["n"]

print(f"세그먼트 수 : {len(seg_fine)} 개")
print(f"가장 높은 세그먼트의 사고율 : {seg_fine['lam_mle'].max():.6f} 건/년"
      f" | 가장 낮은 세그먼트 {seg_fine['lam_mle'].min():.6f}"
      f" | 사고율 0 인 세그먼트 {int((seg_fine['lam_mle'] == 0).sum())} 개")

> **[문제] 미션 2 에서 고친 뒤의 코드가 세그먼트별 사고율을 만드는 방식으로 맞는 것은?**
>
> 1. `.agg(n=("IDpol", "size"))` 가 센 계약 수로 사고 합을 나눈다 — 단위는 건/계약
> 2. 계약별 사고율을 먼저 구한 뒤 세그먼트 안에서 단순 평균한다 — 노출이 긴 계약도 같은 무게다
> 3. `.agg(exposure=("Exposure", "sum"))` 이 쌓은 노출의 합으로 사고 합을 나눈다 — 단위는 건/년

## 242개 세그먼트 중 수축이 크게 적용되는 것을 구분할 수 있을까

### 미션 3 — 242개 세그먼트에 감마 사전분포 적용

#### 할 일 — 미션 3 (25분)

1. 아래 「[직접 해보기] 관측이 적은 세그먼트의 수축」에서 위젯의 슬라이더를 조작해 어느 세그먼트가 크게 이동하는지 먼저 확인합니다.
2. 그 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
3. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
4. 셀을 클릭한 채 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶.
5. 실행 결과(출력)에 나오는 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 4번부터 다시 합니다.

**이 미션의 질문** — 계약 12건에서 사고율이 과대 추정된 세그먼트에, 회사의 과거 경험을 반영하면 값이 어디에 위치합니까?

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `seg_fine` | 세그먼트 242행 표, 열은 `n` · `claims` · `exposure` · `lam_mle` | 고정 |
| `watch` | 점검 대상 세그먼트 R21 × B11 한 행 | 고정 |
| `lam_all` | 전체 계약(회사 전체)의 연간 사고율, 단위 건/년 | 고정 |
| `alpha_0` | 감마 사전분포의 형상모수 α₀, 최빈값을 `lam_all` 에 맞춰 둔 값 | 고정 |
| `beta_0` | 감마 사전분포의 강도 β₀ = 가상 노출 100년 — 위젯 라벨 「사전 강도 β₀」와 같은 값 | 고정 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다.

1. **R21 × B11 세그먼트의 MAP** — 단위 건/년 · 소수점 아래 6자리 · `watch` 한 행에 아래 MAP 공식을 넣은 값
2. **242개 세그먼트 MAP 의 최솟값** — 단위 건/년 · 소수점 아래 6자리 · 242개 전부에 MAP 를 적용한 뒤 가장 작은 값
   - 이 값이 0 보다 크면 사고율이 0 으로 계산되던 세그먼트가 하나도 남지 않았다는 뜻입니다
3. **추가할 한 행** — 같은 세그먼트의 `lam_mle` 와, `lam_mle` 가 0 이던 세그먼트 수
   - 채점 대상은 아니고, 통과 뒤 열리는 완성본과 대조할 항목입니다
4. **그래프 한 장** — 가로축은 세그먼트의 MLE, 세로축은 그 세그먼트의 MAP 인 산점도
   - 점 : 세그먼트 하나에 점 하나, 모두 242개
   - 대각선 : 두 값이 같은 지점을 잇는 $y = x$ 선 하나 — 0 부터 MLE 최댓값까지 긋습니다
   - 가로축 이름 : 「MLE — 데이터가 말한 그대로 (건/년)」
   - 세로축 이름 : 「MAP — 감마 사전분포를 반영한 뒤 (건/년)」

**출력 형식** — 「이름 = 값 단위」 형식으로 1행씩 `print` 합니다. 표를 통째로 출력하면 채점이 어느 값을 읽을지 고르지 못합니다. 그래프는 한 장입니다.

#### 쓸 공식 — 사후분포의 최빈값(mode, MAP)

**감마분포**는 0 보다 큰 값에 정의되는 분포족이고 모수 2개로 모양이 정해집니다. 포아송 사고율의 켤레 사전분포라 오늘의 사전분포를 이 분포로 적습니다. 모수 2개 중 $\beta_0$ 는 **사전분포의 강도**이고 단위는 노출과 같은 년입니다 — 데이터를 보기 전에 가상 노출 $\beta_0$ 년을 이미 관측했다고 두는 값이고, 위젯 라벨 「사전 강도 β₀」가 이 값입니다. $\alpha_0$ 는 그 기간에 관측한 가상 사고 수에 1 을 더한 값입니다.

이론 3절의 **MAP** 는 사후분포의 최빈값을 요율로 쓰는 추정값이었습니다. 3절의 감마 갱신이 분모에 더한 관측 수 $n$ 항에 **노출의 합**을 대입한 것이 이론 5절이고, 그러면 최빈값이 아래 식 하나로 정리됩니다.

$$
\hat{\lambda}_{\text{MAP}} = \frac{\alpha_0 + \sum y_i - 1}{\beta_0 + \sum E_i}
$$

분자는 가상 사고 $\alpha_0 - 1$ 건과 실제 사고 $\sum y_i$ 의 합이고, 분모는 가상 노출 $\beta_0$ 년과 실제 노출 $\sum E_i$ 의 합입니다. 사전분포를 가상의 과거 데이터로 보면 이 식이 해석됩니다. 이론 5절은 $\beta_0 = 0$ 으로 두어 사전분포를 제외했고, 오늘은 그 $\beta_0$ 를 가상 노출 100년으로 되살립니다.

숫자를 넣어 보면 이렇습니다. 「준비 셀」이 $\beta_0 = 100$년으로 설정하고 $\alpha_0$ 를 전체 계약 사고율에 맞춰 11.06 으로 지정하므로, 분자에 미리 더해지는 가상 사고는 $\alpha_0 - 1 = 10.06$건이고 분모에는 가상 노출 100년이 이미 들어 있습니다. 계약 12건인 세그먼트의 실제 노출은 가상 노출 100년에 비해 비중이 매우 작습니다.

그래프의 **대각선**은 가로축 값과 세로축 값이 같은 지점을 잇는 선 $y = x$ 입니다. 그 선 위에 위치한 점이 이동하지 않은 세그먼트이고, 멀어질수록 많이 이동한 세그먼트입니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `seg_fine` · `watch` · `alpha_0` · `beta_0` 은 이미 있으니 다시 만들지 말라고 적습니다.
- **MAP 를 구하는 법** — 위 공식의 분자에 무엇을 더하고 무엇을 빼는지, 분모에 무엇을 더하는지.
- **출력** — 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 가로축·세로축에 무엇을 배치할지 · 대각선의 양 끝 · 가로축·세로축 이름.

**막히면** — AI 튜터에게 「사고가 한 건도 없는 세그먼트에 MAP 공식을 적용하면 분자에 무엇이 남나요」 처럼 물어보세요.

#### 실행 전에 — 예상을 골라 보세요

사전분포는 242개 세그먼트 모두에 똑같이 적용됩니다. 그런데 노출이 1만 년을 넘는 대형 세그먼트의 이동 거리는 얼마입니까?

> **[예측] MLE 에서 MAP 로 옮길 때 노출이 1만 년을 넘는 대형 세그먼트는 소형 세그먼트와 견줘 얼마나 움직일까요?**
>
> 1. 소형 세그먼트와 비슷한 정도로 움직인다
> 2. 소형 세그먼트보다 훨씬 작게, 이동 거리가 0.001 에도 못 미친다
> 3. 아예 움직이지 않는다 — MLE 와 MAP 가 완전히 같은 값이다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

고른 뒤에는 왜 그런지 근거 1문장을 함께 적어 두세요.

<details class="lms-extra"><summary>점검 대상 세그먼트가 R21 × B11 인 이유</summary>

미션 2 의 요율표에는 사고율이 0.00 으로 출력된 세그먼트가 9개 남았습니다. 가장 높은 R43 × B4 는 계약 26건이고, 사고율이 3배 수준으로 계산된 R21 × B11 의 계약은 12건뿐입니다. 관측이 더 적은 쪽을 **점검 대상 세그먼트**로 삼아 `watch` 에 지정해 두었습니다.

</details>

<details class="lms-extra"><summary>가상 노출 100년과 감마 사전분포를 그림으로 읽는 법</summary>

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Poisson_Process.png" width="640" height="361"/>
<p align="center"><em>▲ 사고가 시간에 따라 누적되는 경로를 5회 모의 생성한 그림 — 선의 기울기가 곧 그 관측의 사고율 λ 다. 가로축 눈금 50 까지 가는 동안 보라색(λ = 10)은 약 510건, 파란색(λ = 1)은 50건 남짓이 누적되었다</em><br><span style="font-size:0.9em;color:#656565">출처: Wikimedia Commons, Poisson Process.png, CC0</span></p>

선 하나가 한 번의 관측이고, 범례의 `X(ω_i; t)` 는 i 번째 관측을 시각 t 까지 누적했다는 표기입니다. 우리 사전분포는 기울기가 전체 계약 사고율인 선을 100년까지 미리 그어 둔 것과 같습니다. 그 가상 관측 100년을 사고율 $\lambda$ 의 분포로 옮겨 적은 것이 아래 감마분포 곡선입니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Gamma_distribution_pdf.svg" width="480" height="360"/>
<p align="center"><em>▲ 감마분포는 모수 2개로 모양이 정해지는 분포 가족이다 — k 가 커질수록(검정·초록) 최빈값이 뚜렷해지고, 왼쪽 끝이 가장 높고 오른쪽으로 퍼지는 곡선(빨강·보라)일수록 특정 값을 덜 지지한다</em><br><span style="font-size:0.9em;color:#656565">출처: Wikipedia, Gamma distribution</span></p>

곡선의 k 는 형상모수 $\alpha_0$, θ 는 척도모수 $1/\beta_0$ 입니다. 감마 사전분포에서 곡선의 산포를 정하는 강도는 가상 노출 $\beta_0$ 이고, k = $\alpha_0$ 는 그 기간에 관측한 가상 사고 수에 1 을 더한 값이라 둘이 함께 증가합니다. 우리 사전분포는 k 11 · θ 0.01 인 곡선이라 최빈값이 0.1 부근에 위치합니다.

</details>

### [직접 해보기] 관측이 적은 세그먼트의 수축

아래 위젯은 이론 4절 끝에서 직접 조작해 본 그 화면입니다. 그때는 노출을 늘리며 수축 정도가 0.226 에서 0.006 까지 감소하는 것만 확인했습니다. 이번에는 사전분포의 강도 β₀ 를 100년으로 고정합니다. 위젯 라벨도 「사전 강도 β₀」입니다.

어느 세그먼트가 이동하고 어느 세그먼트가 이동하지 않는지만 다시 확인합니다. 여기서 확인한 이동 패턴이 잠시 뒤 그릴 산점도에서 점 하나하나로 나타납니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Prior,_Likelihood,_Posterior_schematic.svg" width="600" height="200"/>
<p align="center"><em>▲ 사전분포·우도·사후분포 3개의 관계를 가로로 나란히 그린 그림 — 사건이 적게 관측되어 사후분포의 최빈값이 사전분포의 최빈값보다 왼쪽으로 이동했다</em><br><span style="font-size:0.9em;color:#656565">출처: Wikimedia Commons, Prior, Likelihood, Posterior schematic, CC BY-SA 3.0</span></p>

왼쪽은 데이터를 보기 전의 사전분포이고 가로축이 추정하려는 모수입니다. 가운데는 관측된 사건 수 $k$ 가 각 후보를 지지하는 정도인 우도입니다. 세 곡선 중 가운데만 넓이가 1 이 아닙니다 — 우도는 모수의 확률분포가 아니라 후보 지지도 곡선이라 높이를 사전분포·사후분포와 견주지 않습니다. 오른쪽은 둘을 곱해 다시 계산한 사후분포입니다. 사전분포의 강도와 데이터의 양이 나누는 비중이 사후분포의 최빈값 위치를 정하고, 아래 위젯은 그 비중을 직접 바꿔 보는 화면입니다. 이 그림의 모수는 0～1 범위라 우리 사고율 $\lambda$ 와 값의 범위는 다르지만 사전분포·우도·사후분포 3단계의 구조는 같습니다.

#### 위젯에서 확인할 3가지

1. **조작.** 3가지입니다. ① 위쪽 탭 — 그 세그먼트가 `무사고` 인지 `평균 수준` 인지 `사고 많음` 인지. ② `세그먼트 노출 E` 슬라이더 — 3년부터 3,000년까지, 그 세그먼트가 실제로 누적한 관측의 양입니다. ③ `사전 강도 β₀` 슬라이더 — 이번 실습에서는 100년으로 고정합니다. 「준비 셀」이 쓰는 값과 같게 맞춘 설정이므로, 슬라이더 조작은 확인 질문을 마친 뒤에 합니다.
2. **관찰.** 아래 `수축: MLE에서 MAP까지의 이동` 가로 막대에서 MLE 가 MAP 로 얼마나 이동하는지 보세요. 두 점 사이 거리가 오른쪽 `수축 폭 |MLE − MAP|` 입니다. 두 점이 어느 기준선 쪽으로 수축하는지, 그 기준선의 이름이 화면에 무엇으로 적혀 있는지 직접 찾아보세요. 옆에 함께 출력되는 `데이터의 발언권 w = E/(E+β₀)` 도 보세요 — 화면에 적힌 이름은 발언권이지만 뜻은 **데이터 가중치** $w$ 입니다. 잠시 뒤 수축 정도의 기준으로 쓸 **사전분포의 비중**은 그 나머지, 곧 $1 - w$ 입니다. 「가중치」는 이 권에서 데이터 쪽 $w$ 에만 씁니다. 데이터 가중치가 1 에 근접하면 사전분포의 비중이 0 이 되어 MAP 가 MLE 와 같아집니다.
3. **확인 질문.** $E = 3$년일 때와 $E = 3{,}000$년일 때 수축 정도가 왜 이렇게 다른가요? MAP 공식 분모의 두 항 중 어느 쪽이 커지는지를 떠올리면 답이 보입니다.

In [ ]:
# 인터랙티브 위젯 — 셀을 실행하면 나타납니다 (인터넷 연결 필요)
# 아래가 잘려 보이면 height 값을 키우세요.
from IPython.display import IFrame
IFrame("https://aic-data.aiffel.io/api/widget-file/advanced-statistics/map-shrink-lab.html", width=980, height=520)

<details class="lms-answer"><summary>답 확인 — 위젯 조작 뒤 확인 질문의 답</summary>

MAP 공식의 분모에서 사전분포가 차지하는 비중은 $\beta_0 / (\beta_0 + \sum E_i)$ 입니다 — 위젯에서 본 데이터 가중치 $w$ 의 나머지 $1 - w$ 를 우리 기호로 적은 것입니다. 노출이 3년뿐인 세그먼트면 이 비중이 $100/103 \approx 97\%$라 분모의 대부분을 사전분포가 차지하고, MAP 가 0.1006 에 거의 근접합니다.

화면에서 두 점이 수축해 간 기준선에는 `전사 평균 λ = 0.1006` 이라는 이름이 적혀 있습니다. 이 값의 정체는 사전분포의 **최빈값** $(\alpha_0 - 1)/\beta_0$ 입니다. 전체 계약의 연간 사고율과 같아지도록 설정한 값입니다. 사전분포의 **평균** $\alpha_0/\beta_0 = 0.1106$ 은 이와 다른 값이므로 두 요약값을 구분해 읽습니다. 노출이 3,000년이면 같은 비중이 $100/3{,}100 \approx 3\%$로 감소해 MAP 는 MLE 값과 사실상 같아집니다. 이동 거리가 0 이 되는 것은 아니고, 소수점 아래 4자리에서만 변합니다. **수축 정도를 정하는 것은 β₀ 자체가 아니라 사전분포의 비중이고, 그 비중은 관측량에 반비례합니다.**

</details>

<details class="lms-extra"><summary>여러 세그먼트를 한꺼번에 추정할 때의 이득</summary>

<img src="https://en.wikipedia.org/wiki/Special:FilePath/MSE_of_ML_vs_JS.png" width="560" height="456"/>
<p align="center"><em>▲ 여러 세그먼트를 한꺼번에 추정할 때의 오차 — 빨간 수축 추정값 곡선이 파란 MLE 수평선보다 아래에 위치한다</em><br><span style="font-size:0.9em;color:#656565">출처: Wikipedia, MSE of ML vs JS.png, 퍼블릭 도메인</span></p>

세로축 R 은 **평균제곱오차(mean squared error, MSE)** 입니다 — 추정값이 참값에서 벗어난 거리를 제곱해 평균한 값이라, 작을수록 추정이 좋습니다. 빨간 선은 이론 3절의 수축을 여러 세그먼트에 한꺼번에 적용해 얻은 수축 추정값이고, 가로축 ‖θ‖² 는 참값이 사전분포의 평균에서 떨어진 거리의 제곱입니다. MLE 의 MSE 는 참값 위치와 무관하게 10 으로 일정하고, 수축 추정값은 사전분포의 평균 근처에서 1.3 까지 감소했다가 멀어질수록 8 수준으로 증가해 MLE 값에 수렴합니다. 잠시 뒤 그릴 산점도는 이 곡선의 왼쪽 끝과 오른쪽 끝을 242개 세그먼트로 한 화면에 배치한 그림에 해당합니다. 왼쪽 끝이 관측이 적은 세그먼트, 오른쪽 끝이 노출이 큰 세그먼트입니다. 이 MSE 비교는 미션 3 의 계산에 사용하지 않는 확장 내용입니다.

</details>

242개 세그먼트에 적용하면 이 이동이 한꺼번에 발생합니다. 어느 세그먼트가 이동하고 어느 세그먼트가 이동하지 않는지, 미션 3 에서 직접 계산해 그림으로 확인합니다.

In [ ]:
# 미션 3 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 3 에 쓸 변수를 직접 만듭니다
if "df" not in globals():                                       # 이 미션만 따로 열어도 이 셀부터 실행하면 됩니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)       # 어제 정한 전처리 규칙 2개 — 미션 1 과 같습니다
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

group_keys = ["Region", "VehBrand"]                             # 쓸 변수 1 — 세그먼트 기준 열 2개
seg_fine = (df.groupby(group_keys, observed=True)               # 쓸 변수 2 — 242개 세그먼트 MLE 요율표
              .agg(n=("IDpol", "size"), claims=("ClaimNb", "sum"),
                   exposure=("Exposure", "sum"))
              .reset_index())
seg_fine["lam_mle"] = seg_fine["claims"] / seg_fine["exposure"]
lam_all = float(df["ClaimNb"].sum() / df["Exposure"].sum())     # 쓸 변수 3 — 전체 계약의 연간 사고율 (건/년)
beta_0 = 100.0                                                  # 쓸 변수 4 — 사전분포의 유효 노출 = 가상 노출 (년)
alpha_0 = 1 + lam_all * beta_0                                  # 쓸 변수 5 — 최빈값을 전체 계약 사고율에 맞춘 alpha
watch = seg_fine[(seg_fine["Region"] == "R21")                  # 쓸 변수 6 — 미션 3 에서 정한 점검 대상 세그먼트 한 행
                 & (seg_fine["VehBrand"] == "B11")]

print(f"세그먼트 {len(seg_fine)}개 | 컬럼 {list(seg_fine.columns)}")
print(f"전체 계약 사고율 lam_all : {lam_all:.6f} 건/년")
print(f"사전 Gamma(alpha_0={alpha_0:.2f}, beta_0={beta_0:.0f})"
      f" — 이 사전분포의 최빈값 (alpha_0 - 1) / beta_0 = {(alpha_0 - 1) / beta_0:.6f} 건/년")
print(f"점검 대상 세그먼트 watch : {watch.iloc[0]['Region']} x {watch.iloc[0]['VehBrand']}"
      f" | 계약 {int(watch.iloc[0]['n'])}건 · 사고 {int(watch.iloc[0]['claims'])}건"
      f" · 노출 {float(watch.iloc[0]['exposure']):.2f}년 — 한 행짜리 표입니다")
print("쓸 수 있는 이름 :  seg_fine · watch · lam_all · alpha_0 · beta_0")

In [ ]:
# 미션 3 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 사고율이 0.00 으로 출력됐던 세그먼트 9개는 내가 그린 산점도에서 세로축이 0 인 위치에 그대로 남아 있다.**
>
> 1. O (맞다)
> 2. X (틀리다)

> **[문제] 사전분포의 강도를 키우면**
>
> 사전분포 Gamma(α₀, β₀)의 강도 β₀ 를 100년에서 10,000년으로 키우되, 사전분포의 최빈값은 전체 계약 사고율에 그대로 맞도록 α₀ = 1 + λ̂·β₀ 로 다시 계산한다고 합시다. 242개 세그먼트 요율표의 MAP는 어떻게 달라질까요? 관측이 적은 세그먼트와 노출이 큰 세그먼트를 나눠서 예측해 보세요.

## 측정 지표를 사고 건수에서 사고 유무로 바꿔도 같은 절차가 적용될까

> **[과제] 과제 제출 — 무사고 환급 이벤트, 대상 규모 추정**
>
> - **무엇을** 무사고 환급 대상 계약 수 한 문장과 두 지역 비교 한 문장, 242개 세그먼트 산점도 한 장을 만듭니다.
> - **무엇으로** 오늘 사용한 데이터 1개로 수행하고, 준비 셀이 242개 세그먼트 집계와 지역 22개 표까지 만들어 둡니다.
> - **인정** 값 4개와 그림이 모두 채점을 통과한 노트북에 브리핑 3문장과 마케팅팀 답신을 추가한 것입니다.
> - **예** 값 행은 「대상 계약 수 = ○○○,○○○ 건」·「R42 가 R43 보다 높을 확률 = 0.○○○○ 확률」 형태입니다.
> - **지표** 오늘 미션과 달리 사고 유무로 측정하므로 R21 x B11 의 값은 계약 12건 중 사고 계약 2건인 비율이고, 그 비율이 MAP 로 수축됩니다.
> 
> - 「PROMPT 셀」을 실행해 값 4개와 그림 1장이 모두 채점을 통과한 상태로 남깁니다.
> - 그 셀의 PROMPT 는 지우지 말고 내가 쓴 그대로 둡니다.
> - 지역 R43 과 R42 의 사후분포를 비교해 어느 쪽이 더 나쁠 확률을 1행으로 출력합니다.
> - 이 절 끝 양식에 브리핑 3문장과 마케팅팀 답신 3문장을 채웁니다.
>
> 마감: 2026-09-30T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 문제 정의 | 「PROMPT 셀」의 PROMPT 에 목표 · 쓸 변수 · 출력 형식이 내 말로 적혀 있다. | 25 |
> | 검증 설계 | 값 4개와 그림이 모두 채점을 통과했고 브리핑 3문장이 채워져 있다. | 25 |
> | 오류 탐지 | 계약이 적은 세그먼트에서 MLE 와 MAP 가 달라지는 이유를 사전분포의 비중 ν/(ν+n) 으로 설명했다. | 25 |
> | 해석·보고 | 마케팅팀 답신이 대상 계약 수를 단위와 함께 제시하고 3문장으로 끝나며, 지역 비교 결과를 확률과 함께 한 문장으로 적었다. | 25 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 또는 Google Drive 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

### 과제 — 무사고 환급 이벤트, 대상 규모 추정

#### 할 일 — 과제 (25분)

1. 아래 「쓸 공식 — 베타분포와 MAP」을 먼저 읽은 뒤, 그 아래 「준비 셀」을 실행합니다. 준비 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
3. 셀을 클릭한 채 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶.
4. 실행 결과(출력)에 나오는 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.
5. 이 절 끝 양식에 브리핑 3문장과 마케팅팀 답신을 채워 제출합니다.

**이 과제의 질문** — 무사고 환급 이벤트의 대상 계약은 몇 건이고, R42 는 R43 보다 정말 더 나쁩니까?

여기서 측정하는 값인 **사고 경험률**은 계약 하나가 기간 중 사고를 한 번이라도 겪는 비율입니다. 건수가 아니라 사고 발생 여부입니다.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `seg_bin` | 세그먼트 242행 표, 열은 `n` · `s` · `p_mle` | 고정 |
| `p_global` | 전체 계약(회사 전체)의 사고 경험률, 단위 비율 | 고정 |
| `nu` | 베타 사전분포의 가상 계약 수 ν = 100건 — 강도 α_b + β_b 는 102 로 2 만큼 큽니다 | 고정 |
| `reg_bin` | 지역 22행 표, 열은 `n` · `s` · `p_mle` | 고정 |
| `rng` | 표본추출의 난수 시드를 고정해 둔 난수 생성기 | 그대로 부른다 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다.

1. **MLE 로 고른 상위 열 세그먼트의 계약 수** — 단위 건 · 정수
   - 무사고율(= 1 − 사고 경험률)이 높은 순으로 `p_mle` 로 세그먼트 10개를 골라 그 `n` 을 더한 값입니다
2. **MAP 로 고른 상위 열 세그먼트의 계약 수** — 단위 건 · 정수
   - 같은 일을 MAP 열로 한 번 더 합니다. 2개 값이 크게 다르면 명단을 정하는 것이 데이터가 아니라 추정 방법이라는 뜻입니다
   - MAP 열은 아래 「쓸 공식 — 베타분포와 MAP」의 MAP 공식으로 만듭니다
3. **무사고 환급 대상 계약 수** — 단위 건 · 정수
   - `p_global` 을 1 에서 뺀 무사고 비율에 전체 계약 수를 곱합니다. 마케팅팀이 실제로 물어본 숫자가 이 값 1개입니다
4. **R42 가 R43 보다 높을 확률** — 단위 확률 · 소수점 아래 4자리
   - 높다는 것은 사고 경험률 $p$ 가 크다는 뜻이고, 이 확률은 데이터를 고정하고 모수 $p$ 를 무작위로 두는 **사후확률**입니다
   - 두 지역의 사후분포에서 표본 10만 쌍을 추출해 R42 쪽이 더 큰 표본의 비율을 계산합니다
5. **추가할 3개 행** — 채점 대상은 아니고, 통과 뒤 열리는 완성본과 대조할 항목입니다
   - R21 × B11 세그먼트의 MAP, 단위 비율 · 소수점 아래 6자리 — 242개 표에서 지역 R21 · 브랜드 B11 행을 골라냅니다
   - 242개 세그먼트 MAP 의 최솟값, 단위 비율 — 브리핑의 위험 항목 근거입니다
   - 두 지역의 사고 경험률 차이 $p_{R42} - p_{R43}$ 의 95% 신용구간, 단위 비율 — 확률은 차이의 방향을, 신용구간은 차이의 크기까지 나타냅니다
6. **그래프 한 장** — 가로축은 세그먼트의 MLE, 세로축은 그 세그먼트의 MAP 인 산점도
   - 점 : 세그먼트 하나에 점 하나, 모두 242개
   - 대각선 : 가로축 값과 세로축 값이 같은 지점을 잇는 $y = x$ 선 하나, 0 부터 MLE 최댓값까지 곧게 긋습니다
   - 가로축 이름 : 「MLE — 데이터가 말한 그대로 (비율)」
   - 세로축 이름 : 「MAP — 베타 사전분포를 반영한 뒤 (비율)」

**출력 형식** — 값 4개를 「이름 = 값 단위」 형식으로 1행씩 `print` 합니다. 그래프는 한 장입니다.

#### 두 지역 비교 — 고정 규칙 4개

1. 두 지역의 사후분포는 이 과제가 정한 그 베타 사전분포를 지역 표 `reg_bin` 에 그대로 적용해 만듭니다.
2. 확률은 두 사후분포에서 표본 10만 쌍을 추출해 R42 쪽이 더 큰 표본의 비율을 계산합니다(이론 5절 방법 2). 격자 적분(구간을 촘촘한 간격으로 나눠 합을 내는 수치적분)으로 계산해도 같은 값이 나옵니다.
3. 표본추출의 **난수 시드(seed)** 는 「준비 셀」이 `rng` 로 고정해 둡니다. 난수 시드는 난수 생성이 매번 같은 값을 생성하도록 고정해 두는 시작값이라 여러분이 따로 정하지 않습니다.
4. 구간은 같은 표본 10만 쌍에서 두 지역 값의 차이를 계산한 뒤 2.5%·97.5% 백분위수로 냅니다. **신용구간**은 사후분포에서 모수의 값이 95% 확률로 포함되는 구간입니다. 1편의 신뢰구간 95% 는 표집을 반복할 때 구간이 참값을 포함하는 비율이라, 무작위로 두는 대상이 다릅니다.

베타 사전분포를 정하는 공식은 아래 「쓸 공식 — 베타분포와 MAP」에 있습니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `seg_bin` · `p_global` · `nu` · `reg_bin` · `rng` 는 이미 있으니 다시 만들지 말라고 적습니다.
- **사전 설계** — 최빈값을 어디에 맞추고 가상 계약 수를 몇 건으로 설정할지 한 문장으로 명시합니다. 그 문장이 없으면 AI 가 사전분포를 임의로 정합니다.
- **출력** — 값 4개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 가로축·세로축에 무엇을 배치할지 · 대각선의 양 끝 · 가로축·세로축 이름.

**막히면** — AI 튜터에게 「두 지역의 사후 베타분포에서 추출한 값으로 어느 쪽이 큰지를 어떻게 계산하나요」 처럼 물어보세요.

<details class="lms-extra"><summary>배경 — 마케팅팀의 메일, 그리고 측정 지표가 사고 유무로 바뀌는 지점</summary>

미션 3 에서 242개 세그먼트에 감마 사전분포를 적용해 요율표를 안정시켰죠. 그 표를 보고하려는데 마케팅팀이 메일을 보냅니다 — **"사고를 한 번이라도 내는 계약의 비율이 궁금해요. 무사고 환급 이벤트 대상자 규모를 추정해야 해서요."** 건수가 아니라 유무의 문제라, 측정 지표가 이론 4절에서 구분해 둔 다른 쪽인 **사고 경험률**로 옮겨 갑니다.

사고 경험률은 계약 하나가 기간 중 사고를 한 번이라도 겪을 확률이고, 이론에서 $\hat p = 0.0502$ 로 다룬 그 값입니다. 쓰는 도구는 미션 3 과 같고 분포 가족만 감마에서 **베타**로 바꿉니다.

같은 메일 끝에 한 문장이 더 적혀 있습니다 — **"환급 예산을 지역별로 나눠 배정하려는데, R42 가 R43 보다 정말 더 나쁜가요?"** 값 2개 0.0564 와 0.0407 을 나열하는 것만으로는 어느 쪽이 더 나쁜지에 대한 답이 되지 않습니다. 이론 5절은 균등 사전분포로 두 지역의 사후분포를 비교해 0.980 이라는 확률을 산출했습니다. 과제에서는 같은 비교를 회사의 경험을 반영한 사전분포로 다시 계산합니다.

계약 6건인 세그먼트에서 "이 세그먼트는 사고를 안 낸다"는 값이 나와 버리는 큰 변동은 유무로 바꿔도 그대로입니다. 그래서 사전분포도 미션 3 과 같은 원칙으로 정합니다 — 최빈값은 전체 계약의 사고 경험률에 맞추고, 가상 계약 수는 100건으로 둡니다.

</details>

#### 쓸 공식 — 베타분포와 MAP

포아송분포의 켤레 사전분포가 감마분포였듯, 사고 유무의 켤레 사전분포는 **베타분포**입니다. 베타분포는 0 과 1 사이에서만 확률밀도가 정의되는 분포족이고, 모수 2개 $\alpha_b$·$\beta_b$ 로 모양이 정해집니다. 아래 첨자 $b$ 는 「베타 사전분포의 모수」라는 표시로, 미션 3 의 감마 사전분포 모수 $\alpha_0$·$\beta_0$ 와 뜻이 다른 모수이므로 기호를 구분해 씁니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Beta_distribution_pdf.svg" width="480" height="384"/>
<p align="center"><em>▲ 모수 조합이 다른 베타분포 5개 — α 와 β 의 비가 평균 α/(α+β) 를, 둘의 합 α+β 가 강도를 정한다. 최빈값 (α−1)/(α+β−2) 는 따로 읽는다</em><br><span style="font-size:0.9em;color:#656565">출처: Wikimedia Commons, Beta distribution pdf.svg</span></p>

우리가 정할 사전분포 Beta(6.0, 96.0) 은 최빈값이 주황 곡선 α=2·β=5 의 0.2 보다 더 왼쪽인 0.05 에 위치하고 곡선도 더 좁습니다. 범례는 위에서부터 빨강 α=β=0.5, 파랑 α=5·β=1, 초록 α=1·β=3, 보라 α=β=2, 주황 α=2·β=5 순서입니다. 빨강은 양 끝이 높은 U 자, 파랑은 오른쪽 끝으로 갈수록 높아지는 곡선, 초록은 가장 왼쪽이 제일 높고 오른쪽으로 내려가는 곡선, 보라는 최빈값이 0.5 인 좌우대칭 곡선, 주황은 최빈값이 0.2 인 곡선입니다. 가상 관측은 $\alpha_b - 1$ 회 성공·$\beta_b - 1$ 회 실패로 읽습니다 — Beta(6, 96) 은 가상 계약 100건 중 사고 계약 5건을 미리 관측한 것과 같습니다.

**갱신 규칙**도 감마 때와 같은 덧셈입니다. 그 세그먼트의 계약 $n$ 건 중 사고 계약이 $s$ 건이면, **사고 계약 $s$ 는 $\alpha_b$ 에, 무사고 계약 $n - s$ 는 $\beta_b$ 에 그냥 더합니다**. 사후분포는 $\text{Beta}(\alpha_b + s,\ \beta_b + n - s)$ 이고, 그 최빈값을 요율로 쓰는 것이 MAP 입니다.

$$
\hat p_{\text{MAP}} = \frac{\alpha_b + s - 1}{\alpha_b + \beta_b + n - 2}
$$

이론 3절의 베타-베르누이 MAP $\frac{\alpha_0 + k - 1}{\alpha_0 + \beta_0 + n - 2}$ 와 같은 식이고, 성공 횟수 $k$ 항에 그 세그먼트의 사고 계약 수 $s$ 를 대입한 것뿐입니다.

$\alpha_b$·$\beta_b$ 는 사전분포의 모수 2개이고, $s$ 는 그 세그먼트의 사고 계약 수, $n$ 은 그 세그먼트의 계약 수로 단위는 둘 다 건입니다. 분자에서 1 을, 분모에서 2 를 각각 한 번씩 빼는 단계가 최빈값을 구하는 계산입니다. 사후분포의 평균을 읽으면 이 뺄셈 2회가 없습니다.

사전 설계도 미션 3 과 같은 원칙 2개입니다 — **최빈값은 전체 계약의 사고 경험률 $\hat p$ 에 맞추고, 가상 계약 수는 $\nu$ 건으로 지정합니다.** $\nu$ 는 그리스 문자 뉴입니다. 「준비 셀」이 $\nu = 100$ 건으로 지정하고 그 값을 `유효 표본 수 nu` 로 출력하는데, 사전분포의 **강도** $\alpha_b + \beta_b$ 는 그보다 2 큰 102 입니다. 가상 관측 수와 강도가 2 만큼 다른 것은 최빈값 기준으로 모수를 정했기 때문입니다.

<details class="lms-hint"><summary>사전분포 Beta(6.02, 95.98) 이 나오는 유도</summary>

<img src="https://raw.githubusercontent.com/angeloyeo/angeloyeo.github.io/master/pics/2020-01-09-Bayes_rule/pic1.png" width="640" height="305"/>
<p align="center"><em>▲ 베이즈 정리 한 식 — 빨간 상자가 사후 P(H|E), 파란 상자가 사전 P(H) 이고, 과제에서 H 는 그 세그먼트의 사고 경험률 p 다. 분모 P(E) 는 상수라 비례식에서 빠지므로, 과제에서 바꾸는 것은 구조가 아니라 사전분포의 분포족뿐이다</em><br><span style="font-size:0.9em;color:#656565">출처: 공돌이의 수학정리노트, angeloyeo.github.io</span></p>

**가상 계약 수**란 데이터를 보기 전에 계약 $\nu$ 건을 이미 관측했다고 가정한다는 뜻입니다. 위 원칙 2개를 MAP 공식에 대입해 풀면 $\alpha_b - 1 = \hat p\,\nu$, $\beta_b - 1 = (1 - \hat p)\,\nu$ 라는 식 2개가 나옵니다. 전체 계약의 사고 경험률 $\hat p = 0.0502$ 와 $\nu = 100$ 을 대입하면 $\alpha_b = 6.02$·$\beta_b = 95.98$ 입니다. 그러면 $\alpha_b + \beta_b - 2 = \nu$ 가 되어 MAP 는 다음과 같이 정리됩니다.

$$
\hat p_{\text{MAP}} = \frac{\hat p\,\nu + s}{\nu + n}
$$

**가상 계약 $\nu$ 건과 실계약 $n$ 건을 합쳐 사고 경험률을 계산한 값**이라는 뜻입니다. 최빈값 기준으로 읽으면 사전분포의 비중은 정확히 $\nu / (\nu + n)$ 입니다. 사후분포의 평균으로 읽으면 강도를 그대로 쓴 $(\alpha_b + \beta_b)/(\alpha_b + \beta_b + n)$ 이라 값이 조금 다릅니다. 계약 12건인 세그먼트에서는 $100/112$ 로 분모의 약 89%를 사전분포가 차지하고, 계약이 수만 건인 세그먼트에서는 0.2% 미만으로 감소합니다. 미션 3 에서 노출로 계산하던 비중을 계약 수로 바꾼 것뿐입니다.

</details>

In [ ]:
# 과제 · 준비 — 읽고 실행만 합니다. 이 셀이 과제에 쓸 변수를 직접 만듭니다
if "df" not in globals():                                       # 앞 미션을 안 돌렸으면 맨 위 설치 셀과 이 셀을 실행합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)       # 어제 정한 전처리 규칙 2개 — 미션 1 과 같습니다
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

df["HasClaim"] = (df["ClaimNb"] > 0).astype(int)          # 사고를 한 건이라도 냈으면 1, 아니면 0
p_global = float(df["HasClaim"].mean())                   # 쓸 변수 1 — 전체 계약의 사고 경험률 (비율)
nu = 100.0                                                # 쓸 변수 2 — 사전분포의 유효 표본 수 = 가상 계약 수 (건)

# 세그먼트 집계와 MLE 까지는 완성해 두었습니다 (미션 2·3 과 같은 Region x VehBrand 242개)
seg_bin = (df.groupby(["Region", "VehBrand"], observed=True)
             .agg(n=("HasClaim", "size"), s=("HasClaim", "sum"))
             .reset_index())
seg_bin["p_mle"] = seg_bin["s"] / seg_bin["n"]

reg_bin = (df.groupby("Region", observed=True)            # 쓸 변수 3 — 지역 22개 표 (세그먼트 대신 지역)
             .agg(n=("HasClaim", "size"), s=("HasClaim", "sum"))
             .reset_index())
reg_bin["p_mle"] = reg_bin["s"] / reg_bin["n"]
rng = np.random.default_rng(0)                            # 쓸 변수 4 — 표본추출의 난수 시드를 고정합니다

print(f"세그먼트 {len(seg_bin)}개 | 컬럼 {list(seg_bin.columns)}")
print(f"전체 계약의 사고 경험률 p_global : {p_global:.6f} (비율)"
      f" | 사전분포의 유효 표본 수 nu = {nu:.0f} 건")
print(f"사고 경험률이 0 으로 계산된 세그먼트 : {int((seg_bin['p_mle'] == 0).sum())} 개")
print(f"전체 계약 {int(seg_bin['n'].sum()):,}건 — 242개 세그먼트의 계약 수를 모두 더한 값입니다")
print("\n[계약 수가 가장 적은 세그먼트 8개 — 지금은 MLE 만]")
print(seg_bin.nsmallest(8, "n").round(4).to_string(index=False))
_two_regions = reg_bin.set_index("Region")
print(f"\n지역 {len(reg_bin)}개"
      f" | R43 계약 {int(_two_regions.loc['R43', 'n']):,}건 중 사고 계약 {int(_two_regions.loc['R43', 's'])}건"
      f" · R42 계약 {int(_two_regions.loc['R42', 'n']):,}건 중 {int(_two_regions.loc['R42', 's'])}건")
print("\n쓸 수 있는 이름 :  seg_bin (n · s · p_mle 열) · p_global · nu"
      " · reg_bin (n · s · p_mle 열) · rng")

In [ ]:
# 과제 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

**브리핑 3문장** — 아래 3문장을 이 셀에 그대로 채워 제출합니다. 값과 그림이 모두 채점을 통과한 뒤에 씁니다. 만들 것 5번에서 이미 출력한 「242개 세그먼트 MAP 의 최솟값」을 위험 항목의 근거로 씁니다. 그 값이 0 보다 크면 사고 경험률이 0 으로 계산되던 세그먼트가 하나도 남지 않았다는 뜻입니다.

> **결론**: 무사고 환급 대상은 계약 ____ 건(전체 ____ 건의 ____%)으로 추정합니다.
>
> **위험**: ____ (관측이 적은 세그먼트에서 어떤 값을 그대로 쓰면 안 되는지 한 문장)
>
> **코드가 답을 만드는 방식**: ____ (사전분포의 가상 계약 $\nu$ 건과 이 세그먼트의 실계약 $n$ 건을 어떻게 합쳐 비율을 냈는지 한 문장)

**마케팅팀 답신** — 브리핑 바로 아래에 다음 3문장을 채웁니다. 브리핑이 우리 팀에 쓰는 글이라면, 답신은 마케팅팀에 보내는 글입니다.

> **결론**: 무사고 환급 대상은 계약 ____ 건으로 추정됩니다.
>
> **근거**: ____ (이 숫자를 어떤 데이터로 어떻게 냈는지 한 문장)
>
> **한계**: ____ (세그먼트별로 명단을 나눌 때 어떤 값을 그대로 쓰면 안 되는지, 그리고 내가 출력한 「R42 가 R43 보다 높을 확률」이 방향만 말하고 크기는 95% 신용구간이 말한다는 점을 한 문장)

## 오늘의 정리 — 내일은 계약별 요율로

프라이싱 분석가로서 2일차 보고를 정리합니다. 오늘의 목표는 4개였습니다 — 한 숫자 · 242개 세그먼트 요율표 · 수축 · 사고 유무로의 전환입니다. 아래 표는 그 순서대로 정리한 것입니다. 앞 절반의 답은 **노출 1년당 0.1006건**입니다. 다만 그 값을 세그먼트 요율로 적용하면 세그먼트마다 추정값의 표본 변동이 크다는 것까지가 오늘 함께 얻은 답입니다. 그래서 보고서에 포함되는 것은 숫자 하나가 아니라 세그먼트별 MLE 와 MAP, 242개 세그먼트의 추정값 2종입니다.

| 오늘의 질문 | 우리가 얻은 답 |
|---|---|
| 포트폴리오의 연간 사고율은? | 포아송 MLE $\hat{\lambda}$ — 사고 건수의 합을 **노출의 합**으로 나눈 한 숫자 (건/년) |
| 계약당 평균 사고 건수를 쓰면 안 되나? | 노출(평균 0.53년)을 무시한 왜곡 — 연간 기준의 절반쯤으로 과소평가됩니다 |
| 242개 세그먼트 요율표의 문제는? | 관측이 적은 세그먼트에서 MLE 가 0.00 부터 전체 계약 사고율의 3배까지 크게 변동합니다 |
| 어떻게 안정화했나? | 감마 사전분포(가상 노출 100년) × 우도 → 사후분포의 최빈값(MAP)으로 수축 |
| 사고 '유무'로 바꾸면? | 켤레 사전분포를 감마에서 베타로 바꿔도 같은 절차가 성립하는지 — 과제에서 직접 확인합니다. 감마의 강도는 가상 노출 β₀ 이고 베타의 강도는 α+β 입니다 |
| 지역 2개 중 어느 쪽이 더 나쁜가? | 사후분포를 비교해 **확률 하나**로 답합니다 — 과제에서 직접 확인합니다 |

MLE 와 MAP 의 관계는 다음 표와 같습니다.

| | MLE | MAP |
|---|---|---|
| 최대화 대상 | 로그우도 $\ell(\lambda)$ | 로그 사후(정규화 상수 제외) $\ell(\lambda) + \log P(\lambda)$ — 최댓값의 위치만 사후분포와 같습니다 |
| 쓰는 정보 | 데이터만 | 데이터 + 사전지식 |
| 관측이 적은 세그먼트에서 | 표본 변동에 따라 크게 달라짐 | 사전분포 쪽으로 수축해 안정 |
| 관측이 많은 세그먼트에서 | 좋은 추정 | MLE 로 수렴 (사전분포의 영향이 사라짐) |
| 사전분포가 균등분포면(α₀ = 1) | — | 최빈값(MAP)이 MLE 와 동일 — 사후분포의 평균은 다릅니다. 가상 관측 0회지만 강도가 0 인 것은 아닙니다 |

오늘 확정한 것은 2개입니다. 하나는 **전처리 규칙**으로 `ClaimNb` 4건 절사와 `Exposure` 1년 상한이고, 다른 하나는 **242개 세그먼트 요율표의 추정값 2종**인 세그먼트별 MLE 와 MAP 입니다. 이 2개가 내일의 출발점입니다.

## 내일 예고 — 세그먼트 평균으로는 부족하다

오늘 만든 요율표에는 구조적 한계가 있습니다. **같은 세그먼트 안의 계약은 전부 같은 요율을 받는다**는 것. 같은 지역에 살아도 20세 신규 운전자의 스포츠카와 50세 무사고 베테랑의 경차가 같은 보험료를 부과하는 것이 타당할까요? 세그먼트를 더 나누자니 오늘 본 변동이 더 심해질 뿐입니다.

내일은 접근 방법을 바꿉니다. 세그먼트를 나누는 대신 **계약 한 건 한 건의 특성을 입력으로 받아 그 계약만의 사고율 $\lambda_i$ 를 계산하는 회귀 모형**을 만듭니다. 특성은 나이·차량·지역·할증점수입니다. 사용하는 데이터는 오늘과 동일합니다.

내일도 포아송 우도·로그우도 최대화·노출 보정을 그대로 사용합니다. 회귀 모형의 학습이 오늘 배운 MLE 를 계수 여러 개로 확장한 것임을 확인합니다. 내일 만들 첫 모형은 설명변수가 없는 절편만의 모형(intercept-only)인데, 그 예측값이 오늘의 $\hat{\lambda}$ 와 정확히 일치하는지 확인하는 것으로 하루를 시작하겠습니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Regression_de_Poisson.svg" width="480" height="360"/>
<p align="center"><em>▲ 포아송 회귀의 산점도와 적합 곡선 <code>ln(λ) = 2x + 0.5</code> — 계약 특성 <code>x</code> 가 0 에서 1 로 가는 동안 곡선이 나타내는 기대 사고 건수가 1.6건에서 12건으로 증가하니, 오늘 구한 상수 λ̂ 가 내일은 계약마다 값이 달라지는 이 곡선으로 확장된다</em><br><span style="font-size:0.9em;color:#656565">출처: Wikimedia Commons, Regression de Poisson.svg, CC0</span></p>

가로축 <code>Variable explicative x</code> 는 계약이 가진 특성 하나이고, 세로축 <code>Résultats de décompte</code> 는 그 계약에서 관측된 사고 건수입니다(프랑스어로 만든 그림이라 라벨이 프랑스어입니다). 파란 점 하나가 계약 하나이고, 주황 곡선은 특성 <code>x</code> 로부터 그 계약만의 사고율 λ 를 계산해 주는 모형이라, 왼쪽에서 드물고 오른쪽으로 갈수록 잦아지는 이 모양이 곧 요율 차등의 근거가 됩니다.

오늘의 마무리 절차는 다음과 같습니다. 오늘 열린 완성본 풀이와 내 코드가 달랐던 지점 1곳을 떠올린 뒤, 이 절 마지막 퀴즈의 3문장을 채워 팀 채널에 올리고 다른 사람의 답을 읽는 것이 오늘의 마지막 복습입니다. 함께 볼 팀이 없다면 내일의 나에게 남기는 메모로 적어 두어도 효과는 같습니다.

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 모형추정 Day 2 완료</h2>
<div style="color:#656565">어제 적은 우도에 회사의 사전지식을 더해 관측이 적은 세그먼트의 표본 변동을 줄였습니다.<br/>내일은 이 도구로 계약 한 건 한 건의 요율을 계산합니다.</div>
</div>

> **[문제] 요율표 초안을 그대로 사용해도 될까**
>
> 미션 2 가 만든 242개 세그먼트 요율표에는 연간 사고율이 0.00 으로 출력된 세그먼트가 9개 있고, 가장 높은 세그먼트는 전체 계약 사고율의 3배 수준이었습니다. 팀장이 이렇게 말합니다. "0.00 인 세그먼트는 보험료를 최저로 내리고, 3배인 세그먼트는 그대로 3배를 받읍시다." (가) 이 표를 그대로 요율로 쓰면 안 되는 이유를 '증거의 양'이라는 말로, (나) 오늘 그 두 종류의 세그먼트를 어떤 방법으로 고쳤는지, (다) 그 방법이 계약이 수만 건인 대형 세그먼트는 왜 거의 조정하지 않는지 — 3문장으로 적어 보세요.